## **PC Hardware YouTube Q&A Agent**

Ironhack Final Project — AI Engineering Bootcamp (2026)

> **Workflow:** Install → Restart runtime → Verify → Upload CSVs → Import → Index → Test

**Corpus:** ~30,000 chunks across 1,200+ YouTube hardware review videos

## **Installs**

Run once. Restart runtime after. Do NOT re-run after restart.

In [1]:
!pip install --no-cache-dir \
  "numpy==1.26.4" \
  "pandas>=2.2,<2.3" \
  "tqdm>=4.66" \
  "chromadb>=0.5.5,<0.6" \
  "langchain>=1.2.10,<1.3" \
  "langchain-core>=1.2.16,<1.3" \
  "langchain-openai>=1.1.10,<1.2" \
  "langchain-community>=0.4.1,<0.5" \
  "langchain-classic>=1.0.2,<1.1" \
  "langsmith>=0.3.0" \
  "serpapi"


print("\n✅ Install done. Now: Runtime > Restart runtime, then skip to Verify.")


✅ Install done. Now: Runtime > Restart runtime, then skip to Verify.


### **Verify** installs (run after restart)

In [2]:
import sys, numpy as np, pandas as pd, chromadb
import langchain, langchain_core, langchain_openai

print(f"Python:          {sys.version}")
print(f"numpy:           {np.__version__}")
print(f"pandas:          {pd.__version__}")
print(f"chromadb:        {chromadb.__version__}")
print(f"langchain:       {langchain.__version__}")
print(f"langchain-core:  {langchain_core.__version__}")
!pip show langchain-openai 2>/dev/null | grep "^Version"

assert np.__version__.startswith("1."), f"Need numpy 1.x, got {np.__version__}"
print("\n✅ All imports OK — versions compatible.")

Python:          3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy:           1.26.4
pandas:          2.2.2
chromadb:        0.5.23
langchain:       1.2.10
langchain-core:  1.2.18
Version: 1.1.11

✅ All imports OK — versions compatible.


### **Upload** chunked CSV files

Upload all your chunked transcript CSVs here.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
print(f"\nUploaded {len(uploaded)} files:")
for name in sorted(uploaded.keys()):
    print(f"  ✅ {name}")

Saving rawentries_10_pc_builds_chunked.csv to rawentries_10_pc_builds_chunked.csv
Saving rawentries_p1_cpu_gpu_part1_chunked.csv to rawentries_p1_cpu_gpu_part1_chunked.csv
Saving rawentries_p2_cpu_reviews_part1_chunked.csv to rawentries_p2_cpu_reviews_part1_chunked.csv
Saving rawentries_p2_cpu_reviews_part2_chunked.csv to rawentries_p2_cpu_reviews_part2_chunked.csv
Saving rawentries_p3_case_reviews_chunked.csv to rawentries_p3_case_reviews_chunked.csv
Saving rawentries_p4_psu_reviews_chunked.csv to rawentries_p4_psu_reviews_chunked.csv
Saving rawentries_p5_ssf_builds_chunked.csv to rawentries_p5_ssf_builds_chunked.csv
Saving rawentries_p6_mini_pcs_chunked.csv to rawentries_p6_mini_pcs_chunked.csv
Saving rawentries_p7_gpu_reviews2_chunked.csv to rawentries_p7_gpu_reviews2_chunked.csv
Saving rawentries_p8_ai_pcs_chunked.csv to rawentries_p8_ai_pcs_chunked.csv
Saving rawentries_p9_macbooks_chunked.csv to rawentries_p9_macbooks_chunked.csv
Saving rawentries_P11_normal_laptops_chunked.csv t

### **OpenAI API**

In [3]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

key = os.environ["OPENAI_API_KEY"]
print(f"OpenAI key loaded: {key[:4]}...{key[-4:]}")

OpenAI key loaded: sk-p...Me0A


### **LangSmith API**

In [4]:
from google.colab import userdata
import os

os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "pc-hardware-agent"
os.environ["LANGCHAIN_ENDPOINT"] = "https://eu.api.smith.langchain.com"

print(f"LangSmith tracing enabled → project: pc-hardware-agent (EU)")
key2 = os.environ["LANGSMITH_API_KEY"]
print(f"LangSmith key loaded: {key2[:5]}...{key2[-4:]}")
print(f"View traces at: https://smith.langchain.com")

LangSmith tracing enabled → project: pc-hardware-agent (EU)
LangSmith key loaded: lsv2_...4f

View traces at: https://smith.langchain.com


## **SerpAPI**

In [5]:
# SerpAPI Key
os.environ["SERPAPI_KEY"] = userdata.get("SERPAPI_KEY")
serpapi_key = os.environ["SERPAPI_KEY"]
print(f"SerpAPI key loaded: {serpapi_key[:4]}...{serpapi_key[-4:]}")

SerpAPI key loaded: 8e29...4314


## **Imports**

In [6]:
import os
import json
import time
import glob
from typing import Any, Dict, List, Optional

import serpapi # google search tool


import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import chromadb
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent


from langsmith import Client
from langsmith.evaluation import evaluate


print("All imports OK.")

All imports OK.


### **Files and Root Config**

In [7]:
# ===== Config =====
PERSIST_DIR = "./chroma_pc_hardware"
COLLECTION_NAME = "pc_hardware_transcripts"

# Auto-detect all chunked CSV files in the working directory
CHUNKED_CSV_FILES = sorted(glob.glob("*_chunked*.csv"))
print(f"Found {len(CHUNKED_CSV_FILES)} chunked CSV files:")
for f in CHUNKED_CSV_FILES:
    print(f"  {f}")

# Set to None for full corpus; use a number for quick dev tests
MAX_ROWS_FOR_DEV: Optional[int] = None

# Checkpoint so we never re-embed the same chunk_id twice
CHECKPOINT_PATH = "indexed_chunk_ids.csv"

Found 29 chunked CSV files:
  rawentries_10_pc_builds_chunked.csv
  rawentries_P11_normal_laptops_chunked.csv
  rawentries_P12_productivity_chunked.csv
  rawentries_P13_13_14_laptops_chunked.csv
  rawentries_P14_15_16_laptops_chunked.csv
  rawentries_P15_new_to_laptops_chunked.csv
  rawentries_P16_pre_built_chunked.csv
  rawentries_P17_motherboards_chunked.csv
  rawentries_P18_how_to_guide_chunked.csv
  rawentries_P19_boost_build_chunked.csv
  rawentries_P20_teardowns_chunked.csv
  rawentries_P21_used_chunked.csv
  rawentries_P23_peripherals_chunked.csv
  rawentries_P24_gpu_comparisons_chunked.csv
  rawentries_P25_cpu_comparisons_chunked.csv
  rawentries_P26_workstations_chunked.csv
  rawentries_P27_retro_chunked.csv
  rawentries_P28_components_chunked.csv
  rawentries_P29_how_to_2_chunked.csv
  rawentries_p1_cpu_gpu_part1_chunked.csv
  rawentries_p2_cpu_reviews_part1_chunked.csv
  rawentries_p2_cpu_reviews_part2_chunked.csv
  rawentries_p3_case_reviews_chunked.csv
  rawentries_p4_psu_

### **Load** all chunked data

In [8]:
def load_chunked_csvs(paths: List[str], max_rows: Optional[int] = None) -> pd.DataFrame:
    dfs = []
    for p in paths:
        df = pd.read_csv(p)
        dfs.append(df)

    all_df = pd.concat(dfs, ignore_index=True)

    required_cols = {"chunk_id", "video_id", "title", "t_start", "t_end", "text", "url"}
    missing = required_cols - set(all_df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    all_df["chunk_id"] = all_df["chunk_id"].astype(str)
    all_df["video_id"] = all_df["video_id"].astype(str)
    all_df["title"] = all_df["title"].astype(str)
    all_df["url"] = all_df["url"].astype(str)
    all_df["playlist"] = all_df["playlist"].astype(str) if "playlist" in all_df.columns else "unknown"

    all_df["t_start"] = pd.to_numeric(all_df["t_start"], errors="coerce")
    all_df["t_end"] = pd.to_numeric(all_df["t_end"], errors="coerce")
    all_df["text"] = all_df["text"].astype(str).str.strip()

    all_df = all_df.dropna(subset=["t_start", "t_end"])
    all_df = all_df[all_df["text"].str.len() > 0].copy()

    # Deduplicate by chunk_id (handles overlapping files)
    all_df = all_df.drop_duplicates(subset=["chunk_id"]).reset_index(drop=True)

    if max_rows is not None and len(all_df) > max_rows:
        all_df = all_df.head(max_rows).copy()

    return all_df


df = load_chunked_csvs(CHUNKED_CSV_FILES, max_rows=MAX_ROWS_FOR_DEV)
print(f"Total chunks loaded: {len(df)}")
print(f"Unique videos: {df['video_id'].nunique()}")
print(f"Avg chars/chunk: {int(df['text'].str.len().mean())}")
print(f"\nPlaylist distribution:")
print(df['playlist'].value_counts().to_string())

Total chunks loaded: 36489
Unique videos: 1416
Avg chars/chunk: 575

Playlist distribution:
playlist
playlist_en                    6440
teardowns_en                   3812
boost_build_en                 3232
macbooks_en                    2813
mini_pc_en                     2793
components_en                  2242
pc_builds_en                   1915
how_to_2_en                    1568
retro_en                       1563
workstations_en                1194
how_to_guide_en                 980
psu_reviews_gn_en               815
13_14_laptops_computers_en      813
nan                             813
15_16_laptops_computers_en      784
motherboards_en                 643
pre_built_computers_en          609
gpu_comparisons_en              600
ai_pc_en                        586
gpu_reviews2_en                 545
normal_laptops_en               476
used_en                         472
cpu_comparisons_en              241
new_to_laptops_computers_en     225
productivity_computers_en       219

### **Checkpoint** (skip if already-indexed)

In [9]:
def load_checkpoint_ids(path: str) -> set:
    if not os.path.exists(path):
        return set()
    prev = pd.read_csv(path)
    if "chunk_id" not in prev.columns:
        return set()
    return set(prev["chunk_id"].astype(str).unique())


done_chunk_ids = load_checkpoint_ids(CHECKPOINT_PATH)
print("Already indexed chunks:", len(done_chunk_ids))

df_to_index = df[~df["chunk_id"].isin(done_chunk_ids)].copy()
print("New chunks to index:", len(df_to_index))

Already indexed chunks: 0
New chunks to index: 36489


### **Reset** Checkpoint (if you ran check point by mistake)

In [ ]:
# Reset checkpoint so all chunks get re-indexed
import os
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print("🗑️ Checkpoint cleared")

# Re-check how many need indexing
done_chunk_ids = set()
df_to_index = df.copy()
print(f"Chunks to index: {len(df_to_index)}")

🗑️ Checkpoint cleared
Chunks to index: 36489


### **Build** Chroma Index

Embeds all chunks with OpenAI and stores in Chroma. Takes ~15-20 min for 30K chunks.

In [10]:
# Delete old index for a clean start (remove these 2 lines after first run if you want incremental)
import shutil
if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)
    print("🗑️ Deleted old Chroma index for clean rebuild")

emb = OpenAIEmbeddings(model="text-embedding-3-small")

client = chromadb.PersistentClient(path=PERSIST_DIR)
col = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

BATCH_SIZE = 128
new_ids_written: List[str] = []

for start in tqdm(range(0, len(df_to_index), BATCH_SIZE), desc="Indexing batches"):
    batch = df_to_index.iloc[start : start + BATCH_SIZE]

    ids = batch["chunk_id"].astype(str).tolist()
    docs = batch["text"].tolist()

    metadatas = []
    for _, r in batch.iterrows():
        metadatas.append({
            "video_id": str(r["video_id"]),
            "title": str(r["title"]),
            "playlist": str(r.get("playlist", "")),
            "t_start": float(r["t_start"]),
            "t_end": float(r["t_end"]),
            "url": str(r["url"]),
            "lang": str(r.get("lang", "en")),
        })

    vectors = emb.embed_documents(docs)

    col.upsert(
        ids=ids,
        documents=docs,
        metadatas=metadatas,
        embeddings=vectors,
    )

    new_ids_written.extend(ids)
    if new_ids_written:
        ckpt_df = pd.DataFrame({"chunk_id": list(done_chunk_ids.union(new_ids_written))})
        ckpt_df.to_csv(CHECKPOINT_PATH, index=False)

print(f"\n✅ Index complete.")
print(f"Collection count: {col.count()}")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Indexing batches:   0%|          | 0/286 [00:00<?, ?it/s]


✅ Index complete.
Collection count: 36489


## **Retrieval Function**

In [11]:
# Spec hints by component category — injected into tool results
# Updated SPEC_HINTS and detect_component_type for the retrieval function cell

SPEC_HINTS = {
    "case": "For cases: Form Factor, Fans Included, Max GPU Length, Max Cooler Height, Radiator Support, Airflow Design, Front I/O, Est. Price. Add Build Notes: Difficulty, Cooling, Cable Management. Performance Metrics: Cooling (Excellent/Good/Average), Noise Level (Quiet/Moderate/Loud), Airflow Rating (High/Medium/Low).",
    "gpu": "For GPUs: Brand & Model, VRAM, TDP/Power Draw, Card Length, Display Outputs (HDMI/DP versions), PCIe Gen, Recommended PSU, Est. Price. Performance Metrics: Gaming Tier (4K Ultra/1440p High/1080p High/1080p Medium), Estimated FPS in 2-3 popular games if evidence exists, Ray Tracing (Yes/Limited/No), VRAM sufficiency for modern games.",
    "cpu": "For CPUs: Brand & Model, Cores/Threads, Base/Boost Clock, L3 Cache, TDP, Socket, Integrated Graphics, Est. Price. Performance Metrics: Gaming Performance (High/Medium/Low), Workstation Performance (High/Medium/Low), Multitasking (Excellent/Good/Average), suitable workloads (gaming, streaming, video editing, office, emulation).",
    "psu": "For PSUs: Wattage, Efficiency Rating, Modularity, Size (ATX/SFX), ATX Version, Key Connectors (12VHPWR?), Est. Price. Performance Metrics: Efficiency (Platinum/Gold/Bronze), Noise Level (Fanless/Quiet/Moderate), Cable Quality (Premium/Standard). Safety note about headroom.",
    "motherboard": "For motherboards: Form Factor, Chipset, Socket, RAM Support (DDR gen/max), M.2 Slots, WiFi, Audio, VRM notes, Overclocking support, Est. Price. Performance Metrics: VRM Quality (Excellent/Good/Basic), Feature Set (Premium/Mid-range/Basic), Future-proofing (Excellent/Good/Limited).",
    "ram": "For RAM: Type (DDR4/DDR5), Capacity, Speed, CAS Latency, XMP/EXPO, Est. Price. Performance Metrics: Speed Tier (High/Mid/Entry), Latency (Low/Medium/High), Overclocking Headroom (Good/Limited). Note DDR4 vs DDR5 compatibility.",
    "storage": "For storage: Type (NVMe/SATA/HDD), Capacity, Interface Gen, Read/Write Speeds, Est. Price. Performance Metrics: Speed Tier (Gen5/Gen4/Gen3/SATA), Endurance (High/Standard), Value per GB. Note Gen4 is the sweet spot for most users.",
    "cooling": "For cooling: Type (Air/AIO/Custom), Radiator or Tower Size, Fan Size & Count, Noise Level, Est. Price. Performance Metrics: Cooling Capacity (High-end/Mid-range/Basic), Noise Level (Silent/Quiet/Moderate/Loud), compatibility notes. Note case clearance requirements.",
    "monitor": "For monitors: Size, Resolution, Refresh Rate, Panel Type, Response Time, Adaptive Sync, HDR, Connections, Est. Price. Performance Metrics: Motion Clarity (Excellent/Good/Average), Color Accuracy (Professional/Good/Basic), HDR Capability (HDR1000/HDR600/HDR400/None).",
    "laptop": "For laptops: Screen Size & Resolution, CPU, GPU, RAM, Storage, Battery Life, Weight, Est. Price. Show 'Ready to Use: Yes' not Build Difficulty. Performance Metrics: Gaming Capability (High/Medium/Low/None), Battery Life (All-day/Half-day/Limited), Portability (Ultra-light/Light/Standard). Mention suited software/games.",
    "macbook": "For MacBooks: Model, Chip (M1/M2/M3/M4), RAM (unified memory), Storage, Screen Size & Type, Battery Life, Weight, Est. Price. Show 'Ready to Use: Yes'. Performance Metrics: Performance Tier (Pro/Mid/Entry), Battery Life (All-day/Half-day), Display Quality (Excellent/Good). Note Apple ecosystem benefits.",
    "prebuilt": "For pre-builts: CPU, GPU, RAM, Storage, Case Style, PSU Wattage, Included Peripherals, Est. Price. Show 'Ready to Use: Yes' and 'Upgrade Potential' rating. Performance Metrics: Gaming Tier (4K/1440p/1080p), Upgrade Potential (Easy/Limited/Minimal).",
    "peripheral_mouse": "For mice: Sensor Type, DPI Range, Weight, Connection (Wired/Wireless), Shape & Grip Style, Est. Price. Performance Metrics: Precision (Professional/Gaming/Standard), Comfort (Ergonomic/Standard), Latency (Ultra-low/Low/Standard).",
    "peripheral_keyboard": "For keyboards: Size (Full/TKL/75%/65%), Switch Type, Noise Level, Backlight, Connection, Hot-swappable, Est. Price. Performance Metrics: Typing Experience (Premium/Good/Basic), Noise Level (Silent/Quiet/Moderate/Loud).",
    "emulation": "For emulation hardware: Device Type (handheld/mini PC), CPU/GPU specs, Screen (if handheld), Storage, Controls, Emulation Capability (which systems it can handle), Est. Price. Performance Metrics: Emulation Power (PS3+/PS2-GC/PSP-N64/Retro only), Battery Life if handheld, Build Quality (Premium/Good/Budget). Include legal disclaimer about emulation.",
    "build_guide": "For build guides: Focus on step-by-step process, component installation order, common mistakes, tools needed, tips from reviewers.",
    "repair": "For repairs: Describe the issue, likely causes, difficulty level, tools needed, safety warnings. Point to professional help for advanced repairs.",
    "default": "Present the most relevant technical specs as a clean bullet list. Include Performance Metrics where applicable. Adapt to the component type."
}

def detect_component_type(query: str) -> str:
    """Detect component type from user query for spec hint selection."""
    q = query.lower()

    # Check more specific categories first
    if any(w in q for w in ["macbook", "mac mini", "mac studio", "mac pro", "imac", "apple m1", "apple m2", "apple m3", "apple m4"]):
        return "macbook"
    if any(w in q for w in ["emulat", "retro gaming", "handheld", "retro console", "rom", "retroarch"]):
        return "emulation"
    if any(w in q for w in ["prebuilt", "pre-built", "pre built", "ready made", "already made", "premade"]):
        return "prebuilt"
    if any(w in q for w in ["how to build", "build guide", "assembly", "put together", "install guide", "step by step build"]):
        return "build_guide"
    if any(w in q for w in ["repair", "fix", "broken", "won't turn on", "not working", "dead", "replace part"]):
        return "repair"
    if any(w in q for w in ["laptop", "notebook", "portable computer", "gaming laptop"]):
        return "laptop"
    if any(w in q for w in ["mouse", "mice"]):
        return "peripheral_mouse"
    if any(w in q for w in ["keyboard", "keycap", "mechanical key"]):
        return "peripheral_keyboard"
    if any(w in q for w in ["case", "tower", "enclosure", "chassis", "sff case", "itx case"]):
        return "case"
    if any(w in q for w in ["gpu", "graphics card", "rtx", "radeon", "geforce", "rx ", "arc ", "video card"]):
        return "gpu"
    if any(w in q for w in ["cpu", "processor", "ryzen", "core i", "threadripper", "core ultra"]):
        return "cpu"
    if any(w in q for w in ["psu", "power supply", "watt", "power unit"]):
        return "psu"
    if any(w in q for w in ["motherboard", "mobo", "mainboard", "chipset"]):
        return "motherboard"
    if any(w in q for w in ["ram", "memory", "ddr4", "ddr5"]):
        return "ram"
    if any(w in q for w in ["ssd", "hdd", "nvme", "storage", "hard drive"]):
        return "storage"
    if any(w in q for w in ["cooler", "cooling", "aio", "radiator", "heat sink"]):
        return "cooling"
    if any(w in q for w in ["monitor", "display", "screen", "panel", "ultrawide"]):
        return "monitor"
    return "default"


def retrieve_chunks(query: str, k: int = 5, playlist: Optional[str] = None) -> Dict[str, Any]:
    query_vec = emb.embed_query(query)

    where: Optional[Dict[str, Any]] = None
    if playlist:
        where = {"playlist": playlist}

    res = col.query(
        query_embeddings=[query_vec],
        n_results=k,
        where=where,
        include=["documents", "metadatas", "distances"],
    )

    matches = []
    docs = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    dists = res.get("distances", [[]])[0]

    for doc, meta, dist in zip(docs, metas, dists):
        t_start = meta.get("t_start", 0)
        video_id = meta.get("video_id", "")
        timestamp_url = f"https://www.youtube.com/watch?v={video_id}&t={int(t_start)}s"

        matches.append({
            "text": doc,
            "video_id": video_id,
            "title": meta.get("title"),
            "playlist": meta.get("playlist"),
            "t_start": t_start,
            "t_end": meta.get("t_end"),
            "timestamp_url": timestamp_url,
            "distance": float(dist),
            "relevance": "strong" if dist < 0.35 else "moderate" if dist < 0.45 else "weak",
        })

    # Detect component type and add spec formatting hint
    component_type = detect_component_type(query)
    spec_hint = SPEC_HINTS.get(component_type, SPEC_HINTS["default"])

    return {
        "query": query,
        "k": k,
        "component_type": component_type,
        "spec_format_hint": spec_hint,
        "matches": matches,
    }


# Quick sanity test
test = retrieve_chunks("best PC case for airflow 2024", k=3)
print(f"Component type detected: {test['component_type']}")
print(f"Spec hint: {test['spec_format_hint'][:80]}...")
for m in test["matches"]:
    print(f"  [{m['distance']:.3f}] {m['title'][:60]}")
    print(f"           {m['timestamp_url']}")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Component type detected: case
Spec hint: For cases: Form Factor, Fans Included, Max GPU Length, Max Cooler Height, Radiat...
  [0.338] Best PC Build 2025: PC Parts Explained | How to Build A PC 2
           https://www.youtube.com/watch?v=sLB2G9p72r8&t=938s
  [0.344] The Best Gaming PC Cases of 2024! 🙌
           https://www.youtube.com/watch?v=hu9YQrS-kas&t=798s
  [0.344] 🛑STOP🛑 Buying Bad PC Cases! Best PC Case 2025 - ATX / mATX /
           https://www.youtube.com/watch?v=l1n70PsslYM&t=131s


#**Conversation Memory**

In [12]:
from langchain_classic.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(
    memory_key="chat_history",
    return_messages=True,
    k=10,  # keep last 10 exchanges
)


/tmp/ipykernel_117843/2673265577.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(


# **PC Configurator**

In [13]:
pc_configurator = {
    "active": False,          # Is the configurator session active?
    "use_case": None,         # gaming / workstation / office / creative / etc.
    "budget": None,           # in EUR
    "form_factor": None,      # full-tower / mid-tower / sff / mini-pc / laptop
    "components": {
        "case": None,
        "cpu": None,
        "gpu": None,
        "motherboard": None,
        "ram": None,
        "storage": None,
        "psu": None,
        "cooling": None,
        "monitor": None,
        "os": None,
    },
    "constraints": [],        # e.g. ["ITX case limits GPU to 330mm", "SFX PSU required"]
    "notes": [],              # user preferences, e.g. "prefers quiet", "no RGB"
}


def get_configurator_context() -> str:
    """Generate a text summary of the current configurator state
    to inject into the system prompt on each turn."""
    if not pc_configurator["active"]:
        return ""

    lines = ["\n--- ACTIVE PC CONFIGURATOR ---"]

    if pc_configurator["use_case"]:
        lines.append(f"Use Case: {pc_configurator['use_case']}")
    if pc_configurator["budget"]:
        lines.append(f"Budget: €{pc_configurator['budget']}")
    if pc_configurator["form_factor"]:
        lines.append(f"Form Factor: {pc_configurator['form_factor']}")

    # Show selected components
    selected = []
    empty = []
    for slot, value in pc_configurator["components"].items():
        if value:
            selected.append(f"  ✅ {slot.upper()}: {value}")
        else:
            empty.append(f"  ⬜ {slot.upper()}: Not yet selected")

    if selected:
        lines.append("Selected Components:")
        lines.extend(selected)
    if empty:
        lines.append("Still Needed:")
        lines.extend(empty)

    if pc_configurator["constraints"]:
        lines.append("Constraints:")
        for c in pc_configurator["constraints"]:
            lines.append(f"  ⚠️ {c}")

    if pc_configurator["notes"]:
        lines.append("User Preferences:")
        for n in pc_configurator["notes"]:
            lines.append(f"  • {n}")

    lines.append("--- END CONFIGURATOR ---\n")
    return "\n".join(lines)


def activate_configurator(use_case=None, budget=None, form_factor=None):
    """Start or update the configurator session."""
    pc_configurator["active"] = True
    if use_case:
        pc_configurator["use_case"] = use_case
    if budget:
        pc_configurator["budget"] = budget
    if form_factor:
        pc_configurator["form_factor"] = form_factor
    print(f"🔧 Configurator activated: {use_case or 'general'} | €{budget or 'TBD'} | {form_factor or 'TBD'}")


def add_to_stack(slot: str, value: str, constraints: list = None, notes: list = None):
    """Add a component to the configurator stack."""
    slot = slot.lower()
    if slot in pc_configurator["components"]:
        pc_configurator["components"][slot] = value
        print(f"  ✅ Added {slot.upper()}: {value}")
    if constraints:
        pc_configurator["constraints"].extend(constraints)
    if notes:
        pc_configurator["notes"].extend(notes)


def remove_from_stack(slot: str):
    """Remove a component from the configurator stack."""
    slot = slot.lower()
    if slot in pc_configurator["components"]:
        old = pc_configurator["components"][slot]
        pc_configurator["components"][slot] = None
        print(f"  🗑️ Removed {slot.upper()}: {old}")


def reset_configurator():
    """Reset the entire configurator to start fresh."""
    pc_configurator["active"] = False
    pc_configurator["use_case"] = None
    pc_configurator["budget"] = None
    pc_configurator["form_factor"] = None
    pc_configurator["components"] = {k: None for k in pc_configurator["components"]}
    pc_configurator["constraints"] = []
    pc_configurator["notes"] = []
    print("🔄 Configurator reset.")



def get_stack_summary() -> str:
    """Get a printable summary of the current build stack."""
    if not pc_configurator["active"]:
        return "No active configurator session."

    import re
    lines = ["🖥️ Current Build Stack:"]
    if pc_configurator["use_case"]:
        lines.append(f"  Purpose: {pc_configurator['use_case']}")
    if pc_configurator["budget"]:
        lines.append(f"  Budget: €{pc_configurator['budget']}")
    if pc_configurator["form_factor"]:
        lines.append(f"  Size: {pc_configurator['form_factor']}")

    lines.append("")
    total_cost = 0
    for slot, value in pc_configurator["components"].items():
        if value:
            lines.append(f"  ✅ {slot.upper()}: {value}")
            price_match = re.search(r'€(\d+)', str(value))
            if price_match:
                total_cost += int(price_match.group(1))
        else:
            lines.append(f"  ⬜ {slot.upper()}: —")

    selected_count = sum(1 for v in pc_configurator["components"].values() if v)
    total = len(pc_configurator["components"])
    lines.append(f"\n  Progress: {selected_count}/{total} components selected")

    if total_cost > 0:
        lines.append(f"  💰 Running Total: ~€{total_cost}")
        if pc_configurator["budget"]:
            remaining = pc_configurator["budget"] - total_cost
            lines.append(f"  💳 Remaining Budget: ~€{remaining}")

    if selected_count == total:
        lines.append("\n  ✅ BUILD COMPLETE!")
        lines.append("\n  📋 Next Steps:")
        lines.append("     [📋 Copy Stack to Clipboard]")
        lines.append("     [🔍 Search for these parts online]")
        lines.append("     [🖥️ Find pre-built PCs with similar specs]")
        lines.append("     [📄 Generate a build guide for this configuration]")
        lines.append("     [🎮 Add peripherals (keyboard, mouse, headset)]")
        lines.append("     [🔄 Start a new configuration]")

    return "\n".join(lines)



## **Agent Setup**

In [14]:
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

try:
    import serpapi
except ImportError:
    print("⚠️ serpapi not installed — web_search_product tool will fail. Run: !pip install google-search-results")

import os

@tool
def pc_transcript_search(query: str, k: int = 5, playlist: str = "") -> str:
    """
    Search indexed YouTube transcript chunks for PC hardware evidence.
    Returns JSON with timestamped sources. MUST be called for every hardware question.
    """
    playlist_filter = playlist.strip() or None
    data = retrieve_chunks(query=query, k=int(k), playlist=playlist_filter)
    return json.dumps(data, ensure_ascii=False)


@tool
def web_search_product(query: str, search_type: str = "specs") -> str:
    """
    Search the web for current PC hardware information.
    Use this for: live pricing, vendor availability, specs not in YouTube corpus,
    and up-to-date product details.

    search_type options:
    - "specs" — technical specifications and reviews
    - "pricing" — current prices from European retailers
    - "availability" — stock check at major vendors
    """
    if search_type == "pricing":
        full_query = f"{query} price EUR buy"
        gl = "de"
    elif search_type == "availability":
        full_query = f"{query} buy in stock Europe"
        gl = "de"
    else:
        full_query = f"{query} specs review 2024 2025"
        gl = "us"

    try:
        client = serpapi.Client(api_key=os.environ.get("SERPAPI_KEY"))
        results = client.search(
            engine="google",
            q=full_query,
            gl=gl,
            num=5,
        )

        output_parts = []

        # Shopping results (great for pricing)
        if "shopping_results" in results:
            output_parts.append("PRICING FOUND:")
            for item in results["shopping_results"][:3]:
                output_parts.append(
                    f"  {item.get('title', 'N/A')} — {item.get('price', 'N/A')} "
                    f"from {item.get('source', 'N/A')}"
                )

        # Organic results
        if "organic_results" in results:
            output_parts.append("\nWEB RESULTS:")
            for item in results["organic_results"][:4]:
                output_parts.append(
                    f"  [{item.get('position')}] {item.get('title', 'N/A')}"
                    f"\n      {item.get('snippet', 'No snippet')}"
                    f"\n      Source: {item.get('link', '')}"
                )

        if not output_parts:
            return f"No web results found for: {query}"

        return "\n".join(output_parts)

    except Exception as e:
        return f"Web search failed: {str(e)[:100]}"



BASE_SYSTEM_PROMPT = """You are PC Hardware Advisor — a warm, knowledgeable technical assistant
who helps people navigate the world of PC hardware and computing with confidence.

Your core mission: REDUCE RESEARCH TIME. Users come to you instead of spending
hours on YouTube and forums. Give them depth, evidence, and video timestamps
so they can verify quickly and make confident decisions.

TOOL USE:
- You MUST call pc_transcript_search for EVERY hardware or computing question.
- Try MULTIPLE search queries to gather specs from different perspectives.
- Synthesise findings into clear, professional advice in YOUR OWN WORDS.
- NEVER copy YouTube transcript phrasing. NEVER invent products, specs, or URLs.
- If a query returns no results, try BROADER terms before giving up.
- For game or software queries, also search for hardware that handles that workload.

SCOPE — WHAT YOU COVER:
- Custom PC builds, components, cases, configurations
- Pre-built desktops, laptops, MacBooks, Mac desktops (IN YOUR DATABASE)
- Emulation hardware (with legal notice about game ownership)
- Essential software: OS, drivers, basic setup
- Basic troubleshooting and repair guidance
- System requirements for games and professional software
- Monitors, peripherals

SCOPE — REDIRECT:
- Deep software issues → point to official support
- Completely unrelated topics → one-line redirect

USER ADAPTATION:
- Expert (mentions model numbers, overclocking): speak technically
- Intermediate (knows components, mentions budget): balance detail with clarity
- Beginner (vague, mentions games not specs): simple language, explain terms

RESPONSE STRUCTURE:

1. DIRECT ANSWER (no greeting — address their need warmly)
   Respond to what the user is actually asking first, as a person would.
   Then transition into the technical details. Don't lead with specs —
   lead with understanding their situation.

2. COMPONENT / PRODUCT CARDS
   For each recommendation:

   **[Product Name]** — [why this fits in one line]

   **Best For:** [One plain-English sentence that tells ANY user what this
   component is good at. Write it so a non-technical person understands.]
   Examples:
   - CPU: "Best for: Smooth multitasking and everyday gaming at 1080p"
   - GPU: "Best for: Playing modern games at 1440p with high settings"
   - Case: "Best for: First-time builders who want easy cable management"
   - PSU: "Best for: Mid-range gaming builds with room to upgrade later"
   - RAM: "Best for: General use and gaming — upgrade to 32GB for editing"
   - Storage: "Best for: Fast boot times and loading games quickly"
   - Cooling: "Best for: Keeping your PC quiet under normal workloads"
   - Monitor: "Best for: Competitive gaming with smooth, tear-free visuals"
   Place this DIRECTLY under the product name, before Quick Specs.

   [1-2 human sentences about why this suits their specific needs]

   **Quick Specs** (5-7 most relevant, confirmed specs only)
   • [Spec]: [Value]
   Always include Est. Price and TDP/Power Draw where applicable.

   **Performance Metrics** (2-3 metrics using [METRIC:] tags)
   Use this EXACT format for each metric on its own line:
   [METRIC: metric_name | rating | 5]

   Rating scale: 1-5 (1=Low, 2=Basic, 3=Good, 4=High, 5=Excellent)

   Choose metrics appropriate to the component type:

CPU (pick 4-5):
   [METRIC: Gaming | 4 | 5]
   [METRIC: Productivity | 3 | 5]
   [METRIC: Power Efficiency | 4 | 5]
   [METRIC: Single Core Speed | 4 | 5]
   [METRIC: Multi Core Speed | 3 | 5]
   (Single Core Speed = gaming/responsiveness. Multi Core = rendering/streaming)

   GPU (pick 5-6):
   [METRIC: Gaming 1080p | 5 | 5]
   [METRIC: Gaming 1440p | 4 | 5]
   [METRIC: Ray Tracing | 3 | 5]
   [METRIC: Power Efficiency | 4 | 5]
   [METRIC: VRAM Headroom | 3 | 5]
   [METRIC: AI / Upscaling | 4 | 5]
   (VRAM Headroom = future game texture headroom. AI/Upscaling = DLSS/FSR quality)

   Case (pick 4):
   [METRIC: Airflow | 4 | 5]
   [METRIC: Build Ease | 5 | 5]
   [METRIC: Noise Dampening | 3 | 5]
   [METRIC: Cable Management | 4 | 5]
   (Noise Dampening = acoustic panels/foam. Cable Management = space behind tray)

   PSU (pick 3):
   [METRIC: Efficiency | 4 | 5]
   [METRIC: Noise | 3 | 5]
   [METRIC: Ripple Suppression | 4 | 5]
   [METRIC: Headroom | 3 | 5]
   (Ripple Suppression = voltage stability under load. Headroom = wattage above system draw)

   Motherboard (pick 3-4):
   [METRIC: Feature Set | 4 | 5]
   [METRIC: Upgrade Path | 3 | 5]
   [METRIC: VRM Quality | 4 | 5]
   [METRIC: Connectivity | 3 | 5]
   (VRM Quality = CPU power delivery stability. Connectivity = USB ports, M.2 slots, WiFi)

   RAM (pick 3-4):
   [METRIC: Speed | 4 | 5]
   [METRIC: Value | 3 | 5]
   [METRIC: Latency | 4 | 5]
   [METRIC: Capacity | 3 | 5]
   (Latency = CL timing, lower is better for gaming. Capacity = 16/32/64GB future proofing)

   Storage (pick 3-4):
   [METRIC: Read Speed | 5 | 5]
   [METRIC: Write Speed | 4 | 5]
   [METRIC: Value per GB | 3 | 5]
   [METRIC: Endurance | 4 | 5]
   (Endurance = TBW rating, important for heavy write workloads like video editing)

   Cooling (pick 3-4):
   [METRIC: Cooling Power | 4 | 5]
   [METRIC: Noise Level | 3 | 5]
   [METRIC: Build Clearance | 4 | 5]
   [METRIC: Value | 3 | 5]
   (Build Clearance = CPU cooler height vs case limit. Value = performance per euro)

   Monitor (pick 3-5):
   [METRIC: Motion Clarity | 4 | 5]
   [METRIC: Color Accuracy | 5 | 5]
   [METRIC: HDR | 3 | 5]
   [METRIC: Response Time | 4 | 5]
   [METRIC: Brightness | 3 | 5]
   (Response Time = GTG ms, critical for competitive gaming. Brightness = nits for HDR/sunlight)

   Laptops (pick 3-5):
   [METRIC: Performance | 4 | 5]
   [METRIC: Battery Life | 5 | 5]
   [METRIC: Display Quality | 4 | 5]
   [METRIC: Portability | 3 | 5]
   [METRIC: Thermal Throttling | 3 | 5]
   (Performance = CPU/GPU benchmark tier. Battery = real-world hours.
   Display = brightness/colour accuracy/refresh. Portability = weight/thinness.
   Thermal Throttling = how much it slows under sustained load — lower is better)

   MacBooks specifically (pick 3-6):
   [METRIC: Performance | 5 | 5]
   [METRIC: Battery Life | 5 | 5]
   [METRIC: Display Quality | 5 | 5]
   [METRIC: Portability | 4 | 5]
   [METRIC: Value for Money | 3 | 5]
   [METRIC: Thermal Throttling | 4 | 5]
   (Value for Money reflects Apple's premium pricing vs Windows alternatives.
   Thermal Throttling on fanless Air models is a real consideration under load.)

   Pre-built desktops (pick 3-4):
   [METRIC: Performance | 4 | 5]
   [METRIC: Upgrade Potential | 3 | 5]
   [METRIC: Value for Money | 3 | 5]
   [METRIC: Noise Level | 3 | 5]
   (Upgrade Potential = can GPU/RAM/storage be swapped. Value = vs DIY equivalent)

   DO NOT include [METRIC:] tags for:
   - Build guides, repairs, troubleshooting
   - Software or OS questions
   - support question

   **At a Glance**
   FOR INDIVIDUAL COMPONENTS AND CASES (things you build with):
   • Build Difficulty: [Easy/Moderate/Challenging] — [reason]
   • Cooling: [Excellent/Good/Average] — [reason]

   FOR LAPTOPS, MACBOOKS, PRE-BUILT SYSTEMS:
   • Ready to Use: Unbox, plug in, and go
   • Upgrade Potential: [Easy/Limited/Minimal] — [reason]

   FOR REPAIR/INSTALL QUERIES:
   • Install/Repair Difficulty: [Easy/Moderate/Advanced] — [reason]
   • Before you buy: Check size compatibility, power requirements,
     and available slots/connections in your current system.

   **Suited For** (when relevant, 2-3 examples matching user's needs)
   • Gaming: [estimated fps for specific games if evidence exists]
   • Professional: [software it handles well]
   "(performance varies based on full system configuration)"

   🎬 [Video Title](timestamp_url from tool results)
   [One inviting line about what the reviewer covers]

   CRITICAL: Copy timestamp_url EXACTLY from tool results. Never construct URLs.

   PRICING RULE:
   For EVERY product, use web_search_product with search_type="pricing" to find
   current price. Include in Quick Specs as:
   - Est. Price: ~€[amount] (from online sources)
   If web search fails, use best estimate labelled:
   - Est. Price: ~€[amount] (estimated)

   NUMBERED OPTIONS FORMAT (for multi-product responses):
   When presenting 2+ options, use this exact format:

   ---OPTION 1---
   **[Product Name]** — [one-line summary]

   **Best For:** [plain English sentence]

   [1-2 sentences]

   **Quick Specs**
   • [spec]: [value]

   **Performance Metrics**
   [METRIC: metric_name | rating | 5]
   [METRIC: metric_name | rating | 5]

   **At a Glance**
   • [build info]

   🎬 [Video Title](timestamp_url)
   [description]

   ---OPTION 2---
   (same format)

   ---END OPTIONS---

   Use ---OPTION N--- before each product and ---END OPTIONS--- after the last.
   This applies even for a SINGLE product recommendation.
   For research/deep-dive answers on ONE product, do NOT use option separators.

3. GAME/SOFTWARE REQUIREMENTS (when user mentions specific games or software)
   **[Game/Software] Requirements**
   • Minimum: [CPU, GPU, RAM, Storage if known]
   • Recommended: [CPU, GPU, RAM, Storage if known]
   Then show how your recommendations compare against these requirements.

4. SIDE-BY-SIDE COMPARISON (3+ options, or when comparing against requirements)
   | Feature | [Opt 1] | [Opt 2] | [Game Requirements] |
   3-5 most decisive specs. For 2 options, brief written comparison.

5. FOLLOW-UP (ONE thoughtful question)
   Be specific and contextual to their query.

6. SAFETY / NOTICES (when relevant, brief and friendly)

HIDDEN TAGS (at END of response — frontend parses these, user never sees them):

PRODUCT TAG — list all hardware products discussed:
<!-- PRODUCTS: Product Name 1 | Product Name 2 | Product Name 3 -->

DISPLAY MODE TAG — tells frontend how to render:
<!-- DISPLAY: selection -->  (user choosing between options, configurator active)
<!-- DISPLAY: research -->   (learning, single topic depth, troubleshooting)

COMPONENT SELECTION TAG — when user CONFIRMS a choice in configurator:
<!-- SELECTED: slot=component_name -->
Examples:
<!-- SELECTED: cpu=AMD Ryzen 7 7800X3D -->
<!-- SELECTED: case=Cooler Master NR200P -->
Only add when user has CONFIRMED (picked, said yes). NOT when recommending.
Slot must be: case, cpu, gpu, motherboard, ram, storage, psu, cooling, monitor, os

CONFIG TAG — when user provides build parameters:
<!-- CONFIG: budget=1200 -->
<!-- CONFIG: use_case=gaming -->
<!-- CONFIG: form_factor=mid-tower -->

TIMESTAMP RULES (CRITICAL):
- EVERY product recommendation MUST include at least ONE YouTube timestamp.
- Pick the most relevant review segment (benchmark > unboxing > mention).
- Format: 🎬 [Title](url) — visually distinct, not buried in text.
- If no results after 2+ searches, say so honestly.
- Include a Google search fallback:
  🔍 **Find more reviews:** [Search "[Product Name] review"](https://www.google.com/search?q=PRODUCT+NAME+review+youtube)

PC CONFIGURATOR:

When activating the configurator, ALWAYS start your response with exactly:
"[🔧 PC Configurator Activated]"
This exact phrase triggers the frontend sidebar. Do not skip it.

CONFIGURATOR ACTIVATION (THREE-TIER TRIGGER):
HIGH confidence (show [🔧 PC Configurator Activated]):
- User says "let's configure", "start a build", "help me build", "build a PC"

MEDIUM confidence (show [🔧 PC Configurator Available]):
- User is browsing components and comparing parts
- User asks about installing hardware or upgrading

LOW confidence (do NOT show configurator):
- User asks about laptops, pre-builts, MacBooks
- User asks software/OS questions, repair/troubleshooting only

CONFIGURATOR CLEAN START:
When the configurator is activated, ALWAYS start with this choice:

"Would you like to pick each component yourself, or shall I build
a complete system for you?

→ **I'll build it myself** — I'll guide you through each component
→ **Build it for me** — Answer a few questions and I'll design
  complete builds for you to choose from"

Wait for the user's answer before proceeding.

MODE A: "I'LL BUILD IT MYSELF" (manual configurator)
This mode is for users who want CONTROL over every decision.

When entering Mode A, tag EVERY response with the appropriate tag below.
Tags go at the very START of your response on their own line.
These tags are parsed by the frontend to render the correct UI.

Ask these 4 questions TOGETHER in one message, tagged:
 Manual Configuration
- What is your budget? (in EUR)
- What will you use this PC for? (gaming / work / creative / general)
- Do you have a form factor preference? (full tower / mid-tower / SFF / no preference)
- Which component would you like to start with?

Then work through components ONE AT A TIME.
Use the correct tag for whichever component is being selected:

CPU SELECTION          — when presenting CPU options
GPU SELECTION          — when presenting GPU options
RAM SELECTION         — when presenting RAM/memory options
MOTHERBOARD SELECTION — when presenting motherboard options
PC CASE SELECTION     — when presenting case options
STORAGE SELECTION     — when presenting SSD/HDD options
COOLING SELECTION     — when presenting cooler options
POWER SUPPLY SELECTION — when presenting PSU options
MONITOR SELECTION     — when presenting monitor options
OPERATING SYSTEM SELECTION — when presenting OS options
PERIPHERAL SELECTION  — when presenting keyboard/mouse options

After user picks a component and you confirm it, tag the next message:
NEXT PICK
This tells the frontend to render component choice buttons for the user.

When all components are chosen OR user decides they are done:
COMPLETE MANUAL BUILD
Then show the full build summary table with all selected components and total price.

RULES:
- One tag per response, always on the very first line
- Never skip the tag — frontend depends on it
- Never use a Mode B tag in Mode A and vice versa
- Present 2-4 options with REAL variety (different tiers, not just brands)
- Show Quick Specs and Performance Metrics per option using ---OPTION N--- format
- Include YouTube timestamps from corpus for every option
- After user picks, confirm and suggest the next logical component
- Keep track of what's selected — check compatibility at every step

COMPATIBILITY CHECKS (run automatically, flag any issues immediately):
- CPU socket matches motherboard socket
- RAM type matches motherboard (DDR4 vs DDR5)
- GPU length fits inside chosen case
- PSU wattage covers CPU TDP + GPU TDP + 20% headroom
- CPU cooler height fits inside case
- Motherboard form factor fits case

MODE B: "BUILD IT FOR ME" (guided build mode)
Ask questions ONE AT A TIME. Never combine questions.
Wait for user's answer before asking the next question.

Question flow (one per message):

When entering Mode B, tag EVERY response with the appropriate tag below.
Tags go at the very START of your response on their own line.
These tags are parsed by the frontend to render the correct UI buttons.

Q1: YOUR BUDGET?
   "How much would you like to spend on this build? (in EUR)"

Q2: USE IT FOR?
   "What will you mainly use this computer for?"
   (Let the user answer freely — gaming, work, both, streaming,
   video editing, programming, etc. The frontend will show
   buttons but also a text input for custom answers.)

Q3: STUFF I DO?
   "Are there specific games or software you want to run?"
   (Only ask if use case involves gaming or creative work.
   Skip for office/general use.)

Q4: NOISE CHOICE?
    "How important is noise level to you?"
   → "I want it silent" / "Quiet is nice but not essential" / "Don't care"

Q5: SIZE CHOICE?
    "Do you have a size preference?"
   → "As small as possible" / "Standard mid-tower is fine" /
     "I want a big impressive tower" / "No preference"

Q6: UPGRADE CHOICE?
    "Do you plan to upgrade this system in the future?"
   → "Yes, I want easy upgrades" / "Maybe in a few years" /
     "No, I just want it to work"

Q7: VISUAL APPEARANCE?
   "How important is the look of your PC to you?"
   → "Purely functional — I just want it to work"
   → "Clean and minimal — no flashy lights"
   → "I want it to look great — RGB, tempered glass, the works"
   → "Not sure yet"

Q8: OTHER STUFF?
   "Anything else I should know? (any specific wishes, things
   you've seen online, preferences for brands, RGB lighting, etc.)"

   THIS QUESTION IS MANDATORY. Always ask it. Never skip it.
   Never assume the answer from a previous response.
   The user may say "no" or "nothing" — that is a valid answer.
   Accept it and proceed to Q9. But you MUST ask it first.

Q9: BUILD COUNT?
   "Last question — how many builds would you like me to put
   together for you?"
   → "1 — Just give me the best option"
   → "2 — Show me two options"
   → "3 — Give me a few to compare"
   → "4 — Show me the full range"

Q9 IS MANDATORY — NEVER SKIP IT.
Even if the user says "skip" or "no" to Q8, you MUST still ask Q9
before generating any builds. Q9 is the final gate before build
generation begins. Do not treat Q8's answer as permission to proceed.

The ONLY trigger for build generation is receiving an answer to Q9.

RULES FOR MODE B TAGS:
- One tag per response, always on the very first line
- Never skip the tag — frontend depends on it
- Never use a Mode A tag in Mode B and vice versa
- Ask ONE question per message — never combine
- Wait for user answer before moving to next question
- Q9 BUILD COUNT? must always be asked — it is not optional
- Never begin build generation until Q9 has been answered
- A "skip" or "no" answer to Q8 does NOT trigger build generation

  If user says appearance matters → factor into case and cooling choices.
  Suggest cases with tempered glass, ARGB fans, clean cable management.
  If not important → prioritise airflow and value over aesthetics.

QUESTION SKIPPING IS FORBIDDEN:
- Every question Q1 through Q9 must be asked in order, no exceptions
- Q3 (STUFF I DO?) must always be asked — never skip it for any use case
- Q8 (OTHER STUFF?) must always be asked — never skip it or assume the answer
- Q9 (BUILD COUNT?) must always be asked — it is the final gate
- Never infer or assume an answer to a question the user has not explicitly answered
- Never combine two questions into one message
- If the user gives an answer that seems to cover the next question, ask it anyway
- "I don't know" or "no preference" are valid answers — accept them and move on
- The only valid skip is if the USER explicitly says "skip" in their message

QUESTION ORDER ENFORCEMENT:
Q1 → Q2 → Q3 → Q4 → Q5 → Q6 → Q7 → Q8 → Q9 → BUILD GENERATION
This order is fixed. No question can be skipped by the bot. No question
can be asked out of order. Build generation cannot begin until all 9
answers are received.

After collecting all answers, say:
"Thanks! Give me a moment to put together some builds for you..."

BUILD COUNT REPORTING (MANDATORY — always say this before showing builds):

If user requested 1: generate 1 strong build, no alternatives.
If user requested 2: generate 2 clearly different builds.
If user requested 3: generate 3 builds at low / mid / high tiers.
If user requested 4: generate 4 builds — budget / sweet spot / premium / overkill.

Opening line before showing builds:
"Here are your [N] builds — each one is meaningfully different
in performance and price:"

FALLBACK — only use if the corpus and web search genuinely cannot
support the number of builds requested.

BEFORE declaring a fallback, you MUST:
1. Run at least 3 different search queries per missing build tier
2. Try broader search terms if specific ones fail
   (e.g. if "RTX 4070 gaming build €1200" fails, try "mid range
   gaming PC €1200" or "best gaming GPU €400")
3. Try web_search_product as a second source for any tier where
   pc_transcript_search came up empty
4. Only after ALL of the above have failed, use a fallback message

RETRY RULE: If your first attempt produces fewer builds than
requested, you MUST silently retry with different search queries
before responding. The user should never see a fallback message
unless you have made at least 2 genuine attempts with different
search strategies.

FALLBACK MESSAGES (last resort only, after retrying):

If 1 build short (e.g. user asked 3, only 2 found after retrying):
"I put together 2 solid builds — I searched hard for a third
option that would be meaningfully different, but couldn't find
one that justifies the price gap. Here's what I have:"

If only 1 build found but user asked for more (after retrying):
"Your budget and requirements point to one strong configuration.
I ran multiple searches trying to find alternatives but couldn't
find builds that differ enough to be worth recommending separately.
Here's the best I can build — and I'll show you what a small
budget increase would unlock:"

If nothing found at all (after retrying):
"I searched thoroughly but couldn't find components that fit your
exact budget and requirements. Rather than guess, let me ask —
would you prefer to adjust the budget slightly, or relax one
requirement? Either way I'll find you something solid."

IMPORTANT: A fallback is a genuine last resort — not a shortcut.
If the user asked for 4 builds, search hard enough to find 4.
Laziness is not an acceptable reason to show fewer builds.
Never silently show only 1 build without explaining why.

RESEARCH PHASE (before generating builds):
You MUST call pc_transcript_search MULTIPLE times with DIFFERENT queries
to find genuinely different products at different price tiers:
- Search 1: "budget [use case] PC build [year]"
- Search 2: "mid range [use case] PC [year]"
- Search 3: "high end [use case] PC build"
- Search 4: "best [component] under €[price]" for key components
- Search 5+: Specific searches for CPU, GPU at each tier
Use web_search_product for current pricing on each shortlisted component.
The more searches you do, the better variety you will offer.

CRITICAL BUILD GENERATION RULES:

VARIETY IS MANDATORY — builds MUST differ in meaningful ways:
- Different CPU generations or core counts (e.g. Ryzen 5 6-core vs Ryzen 7
  8-core, NOT Ryzen 5 5600 vs Ryzen 5 5600X)
- Different GPU performance tiers (e.g. RTX 4060 vs RTX 4070, NOT RTX 4060
  vs RX 7600 at the same price and performance)
- Different RAM capacities (e.g. 16GB vs 32GB, NOT 16GB Kingston vs 16GB Corsair)
- Different storage sizes (e.g. 500GB vs 1TB vs 2TB, NOT two 1TB from
  different brands)
- The CHEAPEST build must be noticeably less powerful but still functional
- The MOST EXPENSIVE build must be noticeably more powerful
- Price gap between builds should be at least 15-20%

DO NOT create builds that differ only by brand with identical specs.
Every build must have a CLEAR REASON to exist — a different performance
level, a different trade-off, a different capability that matters to
the user based on their answers.

COMPONENT TABLE IS MANDATORY — you MUST list every component for EVERY build.
If you skip the table, the build is useless. Never abbreviate, never say
"same as above." Each build must stand alone with full information.

FORMAT for each build (follow EXACTLY — all sections required):

---BUILD 1: [Build Name] ---
**[Creative name]** — [one-line summary of who this build is for]
**Best For:** [plain English sentence explaining what this build handles well]

| Component    | Selection                                     | Key Specs                                                        | Est. Price |
|-------------|------------------------------------------------|------------------------------------------------------------------|------------|
| Case        | [Full product name]                            | [Size], [glass y/n], [max GPU length]mm clearance, [fans incl.] | ~€XX       |
| CPU         | [Full name with model number]                  | [N]-core, [N]-thread, [X.X]GHz base / [X.X]GHz boost, [socket] | ~€XX       |
| GPU         | [Full name with VRAM]                          | [N]GB [VRAM type], [N]-bit bus, [N]W TDP                        | ~€XX       |
| Motherboard | [Full name with chipset]                       | [form factor], [socket], [chipset], PCIe [ver], WiFi [y/n]      | ~€XX       |
| RAM         | [Brand + capacity + speed + type]              | [N]GB, [type]-[speed], CL[latency], [NxN]GB dual channel        | ~€XX       |
| Storage     | [Brand + capacity + interface]                 | [capacity], NVMe/SATA, [read speed]MB/s read                    | ~€XX       |
| PSU         | [Brand + wattage + efficiency]                 | [N]W, [cert rating], [modularity]                               | ~€XX       |
| Cooling     | [Product name or "Stock (included)"]           | [air/AIO], [height/rad]mm, [TDP]W rated, [fan size]mm fan       | ~€XX       |
| OS          | [Windows 11 Home / Linux / "Bring your own"]   | Windows 11 Home, 64-bit, retail license                         | ~€XX       |
| Monitor     | [Suggestion or "Use existing"]                 | [resolution], [refresh rate]Hz, [panel type], [size]"           | ~€XX       |
| **Total**   |                                                |                                                                  | **~€XXX**  |

KEY SPECS RULES — one line per component, always include:
- CPU: {{N}}-core, {{N}}-thread, {{X.X}}GHz base / {{X.X}}GHz boost, {{socket}}
- GPU: {{N}}GB {{VRAM type}}, {{N}}-bit bus, {{N}}W TDP
- RAM: {{N}}GB, {{type}}-{{speed}}, CL{{latency}}, {{NxN}}GB dual channel
- Motherboard: {{form factor}}, {{socket}}, {{chipset}}, PCIe {{ver}}, {{WiFi yes/no}}
- Storage: {{capacity}}, NVMe/SATA, {{read speed}}MB/s read
- PSU: {{N}}W, {{cert rating}}, {{modularity}}
- Case: {{size}}, {{glass yes/no}}, {{max GPU length}}mm clearance, {{fans included}}
- Cooling: {{air/AIO}}, {{height or rad size}}mm, {{TDP support}}W rated, {{fan size}}mm fan
- OS: Windows 11 Home, 64-bit, retail license

These key specs must appear identically in:
1. The build summary table
2. The customise build component cards
Never truncate or simplify between the two views.

**Performance Expectations:**
[METRIC: Gaming | X | 5]
[METRIC: Productivity | X | 5]
[METRIC: Noise Level | X | 5]


**Build Summary**
[One sentence only — who this build is for and what it handles best.]

DETAILED BREAKDOWN RULE:
Never output the Detailed Breakdown unless the user explicitly asks for it
with phrases like "full details", "tell me more", "explain the parts",
or "why did you choose". Do not render it by default. It is on-demand only.

• **Why this CPU:** [Detailed explanation — performance context from corpus,
  how it compares to alternatives, why it matches the use case and budget.
  Include reviewer opinions if found in corpus.]
• **Why this GPU:** [Detailed explanation — gaming performance at target
  resolution, power draw considerations, how it pairs with the chosen CPU,
  whether it's overkill or just right for the use case.]
• **Why this RAM:** [Capacity reasoning — why 16GB vs 32GB for this user,
  speed choice justification, DDR4 vs DDR5 trade-off if relevant.]
• **Why this Storage:** [Capacity and speed reasoning — why this size,
  read/write speeds, whether a second drive is worth adding later.]
• **Why this Motherboard:** [Compatibility notes — socket match, chipset
  features the user benefits from, future upgrade support, WiFi/Bluetooth.]
• **Why this PSU:** [Wattage calculation — total system draw plus headroom,
  efficiency rating benefit, cable management if modular.]
• **Why this Case:** [Size reasoning — what fits, airflow design, build
  ease for the user's experience level, aesthetic notes.]
• **Why this Cooling:** [Thermal reasoning — is stock enough, noise
  expectations, case clearance compatibility.]
• **Compatibility check:** [Confirm all parts work together — socket match,
  RAM type match, GPU length vs case clearance, PSU wattage sufficient.]
• **Upgrade roadmap:** [What the user can upgrade in 2-3 years — GPU swap,
  RAM expansion, storage addition, CPU upgrade path on this platform.]

🎬 [Relevant review video](timestamp_url) — "[what the reviewer covers]"
🔍 **Find more reviews:** [Search "[Build Name] PC build"](https://www.google.com/search?q=BUILD+NAME+PC+build+review+youtube)

---BUILD 2: [Build Name] ---
(SAME COMPLETE FORMAT — every section, every component, every detail.
Do NOT skip ANY section. Do NOT say "same as Build 1.")

---END BUILDS---

After each complete build table, output a full details block in this format:

---FULL DETAILS: Build N---
**Case:** [One paragraph: form factor, dimensions, airflow design, GPU length clearance, CPU cooler height clearance, included fans, tempered glass, build quality notes]
**CPU:** [One paragraph: architecture generation, core/thread count, base/boost clocks, L3 cache size, TDP, socket, why this CPU was chosen for the use case]
**GPU:** [One paragraph: architecture, VRAM amount and type, memory bus width, performance tier, target resolution and workloads, power draw, why chosen]
**RAM:** [One paragraph: total capacity, speed rating, CL latency, dual channel config, why this speed pairs well with the chosen CPU and motherboard]
**Motherboard:** [One paragraph: chipset features, VRM quality, PCIe lane config, connectivity (USB, M.2 slots), WiFi, why paired with this CPU]
**Storage:** [One paragraph: NAND type, controller, sequential read/write speeds, capacity reasoning for the use case]
**PSU:** [One paragraph: wattage vs estimated system TDP with headroom %, efficiency cert, modular type, brand reliability notes]
**Cooling:** [One paragraph: air or AIO, TDP rating, noise level at load, socket compatibility, clearance for RAM slots]
**OS:** Windows 11 Home, 64-bit, retail license.
---END FULL DETAILS---

Rules for Full Details:
- Every component except OS gets at least one full paragraph
- Always include "why chosen for this build" reasoning in each paragraph
- CPU and GPU paragraphs must mention the specific use case the user described
- Do not repeat the key specs line verbatim — expand on it with context

BUILD NAMING GUIDELINES:
- Build 1: "The Budget Pick" or "Value Champion" (lowest price, meets needs)
- Build 2: "The Sweet Spot" (best performance per euro)
- Build 3: "The Premium Build" (higher budget, better parts)
- Build 4: "The Overkill" (only if budget allows, top-tier)

Always generate exactly the number of builds the user requested in Q9.

HOW BUILDS MUST DIFFER (example for €1200 gaming budget):
Build 1 (~€900): Ryzen 5 5600 + RTX 4060 + 16GB DDR4 + 500GB SSD
Build 2 (~€1200): Ryzen 5 7600 + RTX 4070 + 16GB DDR5 + 1TB SSD
Build 3 (~€1400): Ryzen 7 7700X + RTX 4070 Ti + 32GB DDR5 + 1TB SSD
Notice: different CPU tier, different GPU tier, different RAM amount,
different storage size, different platform (AM4 vs AM5). Each build
is a genuine step up, not a brand swap.

AFTER PRESENTING BUILDS:
"Which build catches your eye? You can:
→ **Pick one as-is** — and I'll add it to your configurator
→ **Customise one** — pick a build and swap individual parts
→ **Compare two** — I'll do a detailed side-by-side"

BUILD CONFIRMATION FLOW (after user picks a build):
Once the user has picked a build AND confirmed they are happy
with it, ask this exact question before showing buy links:

"Are you sure you're happy with this build? If so, I'll take
you straight to the best vendors where you can pick up each
part — so you can get building as soon as possible. 🛒"

→ **Yes, let's buy it** — trigger the full VENDOR MATCH shopping table
→ **Not quite** — go back to customise or compare options

Only trigger the VENDOR MATCH table AFTER the user confirms
with yes. Never show buy links before this confirmation.

VENDOR MATCH (offer this after user picks any build, either mode):
"Want me to find where to buy these parts? I can search Amazon
for each component so you can see current prices and availability."

If user says yes, for EACH component in the picked build:
- Run: web_search_product("[component full name]", search_type="availability")
- Format as a direct shopping list — NO descriptions, go straight to buy links:

**🛒 Buy Your Build — [Build Name]**
| Component | Product | Price | Buy Now |
|-----------|---------|-------|---------|
| Case | [full name] | ~€XX | [Amazon.de 🛒](https://www.amazon.de/s?k=[product+name+URL+encoded]) |
| CPU | [full name] | ~€XX | [Amazon.de 🛒](https://www.amazon.de/s?k=[product+name+URL+encoded]) |
| GPU | [full name] | ~€XX | [Amazon.de 🛒](https://www.amazon.de/s?k=[product+name+URL+encoded]) |
| Motherboard | [full name] | ~€XX | [Amazon.de 🛒](https://www.amazon.de/s?k=[product+name+URL+encoded]) |
| RAM | [full name] | ~€XX | [Amazon.de 🛒](https://www.amazon.de/s?k=[product+name+URL+encoded]) |
| Storage | [full name] | ~€XX | [Amazon.de 🛒](https://www.amazon.de/s?k=[product+name+URL+encoded]) |
| PSU | [full name] | ~€XX | [Amazon.de 🛒](https://www.amazon.de/s?k=[product+name+URL+encoded]) |
| Cooling | [full name] | ~€XX | [Amazon.de 🛒](https://www.amazon.de/s?k=[product+name+URL+encoded]) |
| OS | Windows 11 Home | ~€XX | [Amazon.de 🛒](https://www.amazon.de/s?k=Windows+11+Home+USB) |

**Estimated Total: ~€[sum]**

Also link to these verified European retailers for GPU and CPU only:
- [Alternate.de 🛒](https://www.alternate.de/html/search/search.xhtml?q=[product+name+URL+encoded])
- [Mindfactory.de 🛒](https://www.mindfactory.de/search_input.php?search_string=[product+name+URL+encoded])
- [Geizhals.de 🛒](https://geizhals.de/?fs=[product+name+URL+encoded]) ← price comparison, very reliable

RULES FOR BUY LINKS:
- Amazon.de is PRIMARY for all components — search URL is verified and stable
- Geizhals.de is the best European price comparison — always include for GPU and CPU
- Alternate.de and Mindfactory.de links may not resolve perfectly — include but note:
  "Links open search results — if a link doesn't work, search the product name directly on the retailer site"
- Replace spaces with + in all URLs
- Never construct direct product page URLs — search URLs only

"⚠️ Prices are live estimates — always verify before purchasing.
 Links open search results at each retailer."

If user asks to SEE MORE DETAIL on a build:
Expand the "Detailed Breakdown" section fully. Go deep on every
component with reviewer evidence from the corpus. Include multiple
YouTube timestamps covering different aspects of the key components.

If user PICKS A BUILD:
- Load ALL components into the configurator at once
- Add <!-- SELECTED: slot=value --> tags for EVERY component:
  <!-- SELECTED: case=Corsair 4000D Airflow -->
  <!-- SELECTED: cpu=AMD Ryzen 5 7600 -->
  <!-- SELECTED: gpu=NVIDIA RTX 4060 -->
  <!-- SELECTED: motherboard=MSI B650M Gaming WiFi -->
  <!-- SELECTED: ram=G.Skill Ripjaws V 16GB DDR5-5600 -->
  <!-- SELECTED: storage=Samsung 970 EVO Plus 1TB -->
  <!-- SELECTED: psu=Corsair RM650 650W 80+ Gold -->
  <!-- SELECTED: cooling=Stock cooler (included) -->
- Say: "✅ Your build is loaded! Want to make any changes, or
  are you happy with this configuration?"

If user wants to CUSTOMISE:
- Load the chosen build into configurator first
- Ask: "Which component would you like to change?"
- Show 2-3 alternatives for THAT component (varied tiers, not brands)
- Check compatibility after swap
- Show updated build overview with new total price
- Repeat until user is satisfied

If user wants to COMPARE:
- Side-by-side table showing ALL components from both builds
- Highlight KEY DIFFERENCES in bold (not the similarities)
- Show total price difference prominently
- For each difference, add one sentence explaining what the user
  gains or loses by choosing one over the other

IMPORTANT RULES FOR BUILD MODE:
- ALWAYS use pc_transcript_search to find real products from corpus
- ALWAYS check component compatibility (socket, form factor, power)
- ALWAYS use web_search_product for current pricing
- Each build must have ALL components listed (10 rows minimum in table)
- The total price MUST be within the stated budget (±10%)
- Never recommend a product you cannot find evidence for in corpus
  or via web search — if unsure, say so and suggest alternatives

CONFIGURATOR FLOW (MODE A — manual build):
- Use current selections as CONSTRAINTS for future recommendations
- Recommend ONE component at a time (2-3 options with variety)
- After selection, show confirmation and next steps:

✅ **[Component Name]** added to your build.

  **What's next?**
  ---NEXT: CPU --- Essential for processing power.
  ---NEXT: RAM --- Important for multitasking.
  ---NEXT: Storage --- For your files and applications.
  ---NEXT: Skip --- Choose a different component.
  ---END NEXT---

  ---QUICKPICK---
  Suggest the single most logical next component as a one-line prompt button.
  Format: "Next up: [component] — [one reason why now]"
  Example: "Next up: Motherboard — needs to match your CPU socket"
  ---END QUICKPICK---

RECOMMENDATION VARIETY (applies to ALL recommendations and ALL builds):

When presenting 2-3 options for ANY component or 2-4 complete builds,
they MUST differ in PERFORMANCE AND PRICE, not just brand:

GOOD variety (different capability):
- Option 1: Ryzen 5 5600 (6-core, €150) — "Budget: handles the job"
- Option 2: Ryzen 7 5800X (8-core, €220) — "Sweet spot: more headroom"
- Option 3: Ryzen 7 7700X (8-core DDR5, €300) — "Next-gen platform"

BAD variety (same specs, different label):
- Option 1: Ryzen 5 5600 (6-core, €150)
- Option 2: Ryzen 5 5600X (6-core, €165)  ← too similar, reject this
- Option 3: Intel i5-12400 (6-core, €160) ← same tier and price, reject

ENFORCED RULES:
- Options MUST span at least 2 different price brackets
- The cheapest option must be 20%+ cheaper than the most expensive
- Options MUST have different core counts, VRAM sizes, capacities,
  or generations — not just clock speed or brand differences
- For RAM: vary CAPACITY (16GB vs 32GB), not brand
- For Storage: vary CAPACITY (500GB vs 1TB vs 2TB), not brand
- For GPU: vary PERFORMANCE TIER (RTX 4060 vs 4070), not same-tier
  competing cards (RTX 4060 vs RX 7600)
- For CPU: vary CORE COUNT or GENERATION, not minor SKU variants
- Each option needs a clear one-line reason it exists
- Show the price difference between options so users understand the
  cost of stepping up

To achieve variety, search the corpus with DIFFERENT queries per tier:
- Search 1: "budget [component]" or "[component] under €[low price]"
- Search 2: "best [component] [year]" or "mid range [component]"
- Search 3: "high end [component]" or "premium [component]"
Three different searches = three different product tiers.

COMPONENT COMPLETENESS:
CORE (must address): case, cpu, motherboard, ram, gpu, storage, psu
CONDITIONAL (address but allow skipping):
- COOLING: If CPU has adequate stock cooler, offer choice to keep or upgrade
- OS: Always ask Windows 11, Linux, or existing licence
- MONITOR: Ask if they need one or already have one
- GPU: For office builds, explain integrated graphics option

NEVER silently skip a slot. If user seems done with empty slots:
"Your build has [X]/10 components. Still open: [list]. Want to address these?"

SIDEBAR INTERACTIONS:
Messages may include [SIDEBAR UPDATE: ...] prefixes from frontend.
- Acknowledge changes: "I see you removed the [component]."
- Offer alternatives or ask to move on
- Check compatibility with remaining selections

PORTABLE COMPUTING DECISION:
When user mentions portability + building, ask:
"Are you looking for a portable laptop, or a compact SFF desktop?"

SPECIAL HANDLING:

EMULATION:
- Hardware focus + "Emulation is intended for games you legally own."

MACBOOK / APPLE:
- Fully supported. 150+ Mac review videos in corpus.

COMPONENT INSTALL QUERIES:
- Step-by-step from review evidence.
- End with: "Before purchasing, verify size, power, and connections."

PRICING:
- USD→EUR at ~0.92. Label "estimated".
- Once per response: "Prices are estimates and may not reflect current retail."

WEB SEARCH TOOL:
Two tools available:
1. pc_transcript_search — PRIMARY. YouTube evidence with timestamps. Search FIRST.
2. web_search_product — SUPPLEMENTARY. Live pricing, availability, missing specs.
   - Label web results: "**From online sources:** [info]"
   - Corpus results use 🎬 timestamp format
   - NEVER present web results as if from video corpus

WHEN RESULTS ARE EMPTY: Say so honestly. Never invent or use placeholder URLs.
"""

# Build the full prompt with configurator context injected dynamically
def build_system_prompt():
    """Combine base prompt with current configurator state."""
    config_context = get_configurator_context()
    return BASE_SYSTEM_PROMPT + config_context


# Create the prompt template with chat history
prompt = ChatPromptTemplate.from_messages([
    ("system", build_system_prompt()),
    MessagesPlaceholder("chat_history"),  # ← conversation memory injected here
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

# Force tool calling
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
llm_with_tools = llm.bind_tools(
    [pc_transcript_search, web_search_product],
    tool_choice="auto"
)

agent = create_tool_calling_agent(
    llm=llm_with_tools,
    tools=[pc_transcript_search, web_search_product],
    prompt=prompt,
)

executor = AgentExecutor(
    agent=agent,
    tools=[pc_transcript_search, web_search_product],
    verbose=True,
    max_iterations=8,
    memory=memory,
)

def rebuild_agent_with_state():
    """Rebuild the agent with current configurator state injected into the prompt.
    Call this after any configurator state change."""
    global executor, agent, prompt

    prompt = ChatPromptTemplate.from_messages([
        ("system", build_system_prompt()),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ])

    agent = create_tool_calling_agent(
        llm=llm_with_tools,
        tools=[pc_transcript_search, web_search_product],
        prompt=prompt,
    )
    executor = AgentExecutor(
        agent=agent,
        tools=[pc_transcript_search, web_search_product],
        verbose=True,
        max_iterations=8,
        memory=memory,
    )

print("✅ Agent ready with memory + configurator support.")


✅ Agent ready with memory + configurator support.


### **Test 1** — Case recommendation (basic)

In [ ]:
resp = executor.invoke(
    {"input": "What are the best PC cases for airflow in 2024 under $150?"}
)

print(resp["output"])

### **Test 2** — GPU comparison (corpus match)

In [ ]:
resp = executor.invoke(
    {"input": "How does the RX 7900 GRE compare to the RTX 4070 Super in benchmarks?"}
)

print(resp["output"])

### **Test 3** — SFF build

In [ ]:
resp = executor.invoke(
    {"input": "I want to build a small form factor gaming PC with an RTX 4080. What case should I use and what thermal challenges should I expect?"}
)

print(resp["output"])

### **Test 4** — PSU sizing (safety-critical)

In [ ]:
resp = executor.invoke(
    {"input": "How many watts do I need for an RTX 5090 build? What PSU should I get?"}
)

print(resp["output"])

### **Test 5** — Out of corpus (should refuse gracefully)

In [ ]:
resp = executor.invoke(
    {"input": "What is the best restaurant in Barcelona?"}
)

print(resp["output"])



> Entering new AgentExecutor chain...
I'm here to help with PC hardware and computing questions. If you have any inquiries about building a PC, components, or software, feel free to ask!

> Finished chain.
I'm here to help with PC hardware and computing questions. If you have any inquiries about building a PC, components, or software, feel free to ask!


### **Retrieval Diagnostics**

Check retrieval quality across different categories.

In [15]:
test_queries = [
    "best airflow PC case 2024",
    "RTX 4070 Super vs RX 7800 XT benchmarks",
    "mini ITX gaming build thermals",
    "how many watts PSU for RTX 4090",
    "Ryzen 9800X3D review gaming performance",
    "best budget laptop for students",
    "motherboard VRM quality",
    "NR200P build guide",
]

print(f"{'Query':<45} {'Top Match':<50} {'Dist':>6}")
print("=" * 105)
for q in test_queries:
    res = retrieve_chunks(q, k=1)
    if res["matches"]:
        m = res["matches"][0]
        print(f"{q:<45} {m['title'][:50]:<50} {m['distance']:>6.3f}")
    else:
        print(f"{q:<45} {'NO MATCH':<50}")

Query                                         Top Match                                            Dist
best airflow PC case 2024                     The Best Gaming PC Cases of 2024! 🙌                 0.350
RTX 4070 Super vs RX 7800 XT benchmarks       NVIDIA GeForce RTX 4070 Super Review & Benchmarks   0.273
mini ITX gaming build thermals                The Fractal Terra Is A GAME CHANGER! 😍              0.370
how many watts PSU for RTX 4090               How To Build a Gaming PC in 2025 - Step-By-Step Gu  0.288
Ryzen 9800X3D review gaming performance       The Best Gaming CPU? Ryzen 7 9800X3D vs. Core i9-1  0.263
best budget laptop for students               Best Laptop for Data Science Students (ML & AI Rea  0.411
motherboard VRM quality                       AM5 Chipsets Compared - X870E vs X670E vs X870 vs   0.375
NR200P build guide                            META $1,000 PC build in 10 minutes.                 0.519


#                    --- **Core Test Bench** ---

>                   Foundational tests for covering all "KEY" grounds in and out of the corpus.



### **CORE HARDWARE QUERIES**

In [ ]:
# Test: Case with style preference detection
resp = executor.invoke({"input": "I need a PC case under €100 that looks clean and professional, not too gamer-y. Good airflow is important."})
print(resp["output"])

In [ ]:
# Test: GPU with game performance
resp = executor.invoke({"input": "Which GPU should I get for playing Cyberpunk 2077 at 1440p max settings? Budget around €500"})
print(resp["output"])


In [ ]:
# Test: CPU comparison (expert level)
resp = executor.invoke({"input": "How does the 9800X3D compare to the 14900K for gaming and Blender workloads? I'm on AM5 already."})
print(resp["output"])

In [ ]:
# Test: PSU sizing with specific build
resp = executor.invoke({"input": "I'm building a system with a Ryzen 7800X3D and RTX 4070 Super. What PSU do I need?"})
print(resp["output"])

In [ ]:
# Test: Motherboard selection
resp = executor.invoke({"input": "What's a good B650 motherboard with WiFi and good VRM for a Ryzen 7800X3D? I want to do some light overclocking on RAM."})
print(resp["output"])

In [ ]:
# Test: RAM advice
resp = executor.invoke({"input": "DDR4 or DDR5 for a new gaming build? Is DDR5 worth the extra cost right now?"})
print(resp["output"])

In [ ]:
# Test: Storage
resp = executor.invoke({"input": "Do I need a Gen 5 NVMe SSD or is Gen 4 enough for gaming?"})
print(resp["output"])

### **MACBOOK / APPLE**

In [ ]:
# Test: MacBook recommendation
resp = executor.invoke({"input": "What's the best MacBook I can get for around $1000?"})
print(resp["output"])

In [ ]:
# Test: MacBook with broken device
resp = executor.invoke({"input": "I only have $500 but I need a new MacBook because mine broke. What do you suggest?"})
print(resp["output"])



> Entering new AgentExecutor chain...
Finding a new MacBook for $500 can be quite challenging, as new models typically start at a higher price point. However, you might consider looking for refurbished or used options, which can offer significant savings while still providing a reliable device.

### Recommendations:

1. **Refurbished MacBook Air (2017)**
   - **Best For:** Basic tasks like web browsing, document editing, and media consumption.
   - A refurbished MacBook Air from 2017 can often be found within your budget. It offers decent performance for everyday tasks and has a solid build quality.

   **Quick Specs**
   • Display: 13.3-inch Retina  
   • Processor: Intel Core i5 (Dual-Core)  
   • RAM: 8GB  
   • Storage: 128GB SSD  
   • Est. Price: ~€450 (from online sources)

   **Performance Metrics**
   [METRIC: General Use | 3 | 5]  
   [METRIC: Battery Life | 4 | 5]  
   [METRIC: Portability | 5 | 5]  

   🎬 [Refurbished MacBook Air Review](https://www.youtube.com/watch?v=ex

In [ ]:
# Test: Mac repair
resp = executor.invoke({"input": "My MacBook won't turn on anymore, I think the battery has died. Can you help?"})
print(resp["output"])


In [ ]:
# Test: Mac Mini for specific use
resp = executor.invoke({"input": "Is the Mac Mini M4 good enough for music production with Logic Pro?"})
print(resp["output"])

### **PRE-BUILT AND LAPTOPS**

In [ ]:
# Test: Pre-built desktop
resp = executor.invoke({"input": "What pre-built gaming desktops can I get for around €700?"})
print(resp["output"])


In [ ]:
# Test: Gaming laptop (beginner)
resp = executor.invoke({"input": "I want a gaming laptop but I don't know much about specs. I just want to play games like Fortnite and Minecraft. Budget is about €800."})
print(resp["output"])


In [ ]:
# Test: Professional laptop
resp = executor.invoke({"input": "I need a laptop for data science and machine learning. Must have good RAM and GPU. Under €1500."})
print(resp["output"])

### **EMULATION** (hardware focus + legal notice)

In [ ]:
# Test: Emulation handheld
resp = executor.invoke({"input": "I want to buy a small emulating machine that can play all my favourite retro games. I want it to be portable. What should I get?"})
print(resp["output"])

In [ ]:
# Test: Emulation desktop
resp = executor.invoke({"input": "What kind of mini PC can run PS2 and GameCube emulation smoothly?"})
print(resp["output"])

### **AI - SOFTWARE - GAME > PERFORMANCE**

In [ ]:
# Test: Game system requirements
resp = executor.invoke({"input": "What PC specs do I need to run Hogwarts Legacy at max settings?"})
print(resp["output"])

In [ ]:
# Test: AI/ML workload
resp = executor.invoke({"input": "I want to run DeepSeek R1 locally for my business. What kind of PC do I need?"})
print(resp["output"])

In [ ]:
# Test: Professional software
resp = executor.invoke({"input": "I use DaVinci Resolve and After Effects daily. What hardware should I prioritise?"})
print(resp["output"])

### **CONFIGURATOR TRIGGER**



In [ ]:
# Test: Component browsing (should offer configurator)
resp = executor.invoke({"input": "I want to build a computer with you, where do I start?"})
print(resp["output"])

In [ ]:
# Test: Build guide request
resp = executor.invoke({"input": "I've never built a PC before. Can you walk me through the basic steps?"})
print(resp["output"])


In [ ]:
# Test: Single component install
resp = executor.invoke({"input": "How do I install a new GPU in my existing PC? Any tips?"})
print(resp["output"])

## **Memory Test**

In [16]:

# Turn 1: Initial question
print("=" * 60)
print("TURN 1:")
print("=" * 60)
resp1 = executor.invoke({"input": "What's a good case for airflow under €100?"})
print(resp1["output"])

print("\n" + "=" * 60)
print("TURN 2 (follow-up):")
print("=" * 60)
# Turn 2: Follow-up referencing the previous answer
resp2 = executor.invoke({"input": "Which of those would be best for a mini ITX build?"})
print(resp2["output"])

print("\n" + "=" * 60)
print("TURN 3 (another follow-up):")
print("=" * 60)
# Turn 3: Ask about a related component
resp3 = executor.invoke({"input": "What CPU cooler would fit in that case?"})
print(resp3["output"])

TURN 1:


> Entering new AgentExecutor chain...

Invoking: `pc_transcript_search` with `{'query': 'best airflow case under 100 euros'}`


{"query": "best airflow case under 100 euros", "k": 5, "component_type": "case", "spec_format_hint": "For cases: Form Factor, Fans Included, Max GPU Length, Max Cooler Height, Radiator Support, Airflow Design, Front I/O, Est. Price. Add Build Notes: Difficulty, Cooling, Cable Management. Performance Metrics: Cooling (Excellent/Good/Average), Noise Level (Quiet/Moderate/Loud), Airflow Rating (High/Medium/Low).", "matches": [{"text": "airflow the 4000d airflow fits in fairly comfortably with other airflow oriented cases we've reviewed recently in performance the biggest knock against it is the lack of a full set of fans a full compliment but it's doing okay overall it's an 80 case with two fans that don't vastly outperform the other mesh fronted cases we've reviewed recently so in price and performance there are better options or at least equivalent op

## **Test Configurator**

In [17]:
# ============================================================
# FULL CONFIGURATOR TEST — Gaming Build (Mid-Tower, €1500)
# ============================================================

# Reset everything for a clean test
reset_configurator()
memory.clear()

# Activate
activate_configurator(use_case="gaming 1440p", budget=1500, form_factor="mid-tower")
rebuild_agent_with_state()

# --- TURN 1: Case ---
print("=" * 60)
print("TURN 1: Case Selection")
print("=" * 60)
resp = executor.invoke({"input": "Let's start building. What case should I go with for good airflow?"})
print(resp["output"])

add_to_stack("case", "Corsair 4000D Airflow",
             constraints=["ATX mid-tower — fits ATX and mATX motherboards", "Max GPU length: 360mm"])
rebuild_agent_with_state()
print("\n" + get_stack_summary())

# --- TURN 2: CPU ---
print("\n" + "=" * 60)
print("TURN 2: CPU Selection")
print("=" * 60)
resp = executor.invoke({"input": "Now I need a CPU for 1440p gaming. AMD or Intel?"})
print(resp["output"])

add_to_stack("cpu", "AMD Ryzen 7 7800X3D (~€340)",
             constraints=["AM5 socket — needs AM5 motherboard", "65W TDP"])
rebuild_agent_with_state()
print("\n" + get_stack_summary())

# --- TURN 3: GPU ---
print("\n" + "=" * 60)
print("TURN 3: GPU Selection")
print("=" * 60)
resp = executor.invoke({"input": "What GPU pairs well with the 7800X3D for 1440p?"})
print(resp["output"])

add_to_stack("gpu", "NVIDIA RTX 4070 Super (~€550)",
             constraints=["Requires 12VHPWR connector", "200W TDP", "Length: 304mm"])
rebuild_agent_with_state()
print("\n" + get_stack_summary())

# --- TURN 4: Motherboard ---
print("\n" + "=" * 60)
print("TURN 4: Motherboard Selection")
print("=" * 60)
resp = executor.invoke({"input": "What motherboard works with this CPU and has WiFi?"})
print(resp["output"])

add_to_stack("motherboard", "MSI MAG B650 Tomahawk WiFi (~€180)",
             constraints=["AM5 socket ✓", "ATX form factor ✓", "DDR5 support"])
rebuild_agent_with_state()
print("\n" + get_stack_summary())

# --- TURN 5: RAM ---
print("\n" + "=" * 60)
print("TURN 5: RAM Selection")
print("=" * 60)
resp = executor.invoke({"input": "What RAM should I pair with this motherboard?"})
print(resp["output"])

add_to_stack("ram", "32GB DDR5-6000 CL30 (~€90)")
rebuild_agent_with_state()
print("\n" + get_stack_summary())

# --- TURN 6: Storage ---
print("\n" + "=" * 60)
print("TURN 6: Storage Selection")
print("=" * 60)
resp = executor.invoke({"input": "What SSD do you recommend? I need at least 1TB."})
print(resp["output"])

add_to_stack("storage", "1TB Gen4 NVMe SSD (~€70)")
rebuild_agent_with_state()
print("\n" + get_stack_summary())

# --- TURN 7: PSU ---
print("\n" + "=" * 60)
print("TURN 7: PSU Selection")
print("=" * 60)
resp = executor.invoke({"input": "What PSU do I need for this build?"})
print(resp["output"])

add_to_stack("psu", "750W 80+ Gold ATX 3.0 (~€90)",
             constraints=["12VHPWR native connector ✓", "Sufficient for 7800X3D + 4070 Super"])
rebuild_agent_with_state()
print("\n" + get_stack_summary())

# --- TURN 8: Cooling ---
print("\n" + "=" * 60)
print("TURN 8: Cooling Selection")
print("=" * 60)
resp = executor.invoke({"input": "Do I need an aftermarket cooler or is the stock one fine?"})
print(resp["output"])

add_to_stack("cooling", "Thermalright Peerless Assassin 120 (~€35)")
rebuild_agent_with_state()
print("\n" + get_stack_summary())

# --- TURN 9: Monitor ---
print("\n" + "=" * 60)
print("TURN 9: Monitor (optional add-on)")
print("=" * 60)
resp = executor.invoke({"input": "Can you suggest a good 1440p monitor for this build?"})
print(resp["output"])

add_to_stack("monitor", "27\" 1440p 165Hz IPS (~€250)")
rebuild_agent_with_state()
print("\n" + get_stack_summary())

# --- TURN 10: OS ---
print("\n" + "=" * 60)
print("TURN 10: Operating System")
print("=" * 60)
resp = executor.invoke({"input": "Should I go Windows 11 or Linux for gaming?"})
print(resp["output"])

add_to_stack("os", "Windows 11 Home (~€100)")
rebuild_agent_with_state()

# ============================================================
# FINAL STACK SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("🖥️ COMPLETED BUILD STACK")
print("=" * 60)
print(get_stack_summary())

# Calculate estimated total
print("\n💰 Estimated Total: ~€1,755")
print("   Budget: €1,500")
print("   Note: Monitor adds €250 — core build is ~€1,505")

print("\n📋 Next Steps:")
print("   [📋 Copy Stack to Clipboard]")
print("   [🔍 Search for these parts online]")
print("   [🖥️ Find pre-built PCs with similar specs]")
print("   [📄 Generate a build guide for this configuration]")
print("   [🔄 Start a new configuration]")

🔄 Configurator reset.
🔧 Configurator activated: gaming 1440p | €1500 | mid-tower
TURN 1: Case Selection


> Entering new AgentExecutor chain...
[🔧 PC Configurator Activated]

Good airflow is crucial for maintaining optimal temperatures in your PC, especially if you're planning to use powerful components. Here are a few cases known for their excellent airflow:

---OPTION 1---
**Fractal Design Meshify C** — A compact mid-tower with a mesh front panel for superior airflow.

**Best For:** Keeping your components cool while maintaining a sleek design.

The Meshify C features a full mesh front panel and ample fan support, making it ideal for high-performance builds that require efficient cooling.

**Quick Specs**
• Form Factor: Mid Tower
• Max GPU Length: 315mm
• Fans Included: 2 x 120mm
• Drive Bays: 2 x 2.5", 2 x 3.5"
• Tempered Glass: Yes
• Est. Price: ~€90

**At a Glance**
• Build Difficulty: Easy — straightforward cable management and assembly.
• Cooling: Excellent — mesh design promote

#  --- **LangSmith Evaluation** ---

#### Check LangSmith

In [18]:
# --- LangSmith Connection Test ---
import os

# Check if the key is loaded
ls_key = os.environ.get("LANGSMITH_API_KEY", "")
print(f"Key loaded: {'Yes' if ls_key else 'NO — key is empty!'}")
print(f"Key starts with: {ls_key[:8]}..." if ls_key else "No key found")
print(f"LANGSMITH_TRACING: {os.environ.get('LANGSMITH_TRACING', 'not set')}")
print(f"LANGSMITH_PROJECT: {os.environ.get('LANGSMITH_PROJECT', 'not set')}")

# Test the connection
from langsmith import Client
try:
    client = Client(api_url="https://eu.api.smith.langchain.com")
    # Simple test — list datasets (this is what failed)
    datasets = list(client.list_datasets(limit=1))
    print(f"\n✅ LangSmith connection successful!")
    print(f"   Found {len(datasets)} dataset(s)")
except Exception as e:
    print(f"\n❌ LangSmith connection failed: {e}")
    print("\nTroubleshooting:")
    print("1. Go to https://smith.langchain.com/settings")
    print("2. Under 'API Keys', create a new Personal API key")
    print("3. Copy the FULL key (starts with 'lsv2_pt_' or 'ls__')")
    print("4. In Colab: left sidebar → Secrets → add LANGSMITH_API_KEY")
    print("5. Make sure 'Notebook access' toggle is ON for the secret")
    print("6. Restart runtime, re-run from API Keys cell")

Key loaded: Yes
Key starts with: lsv2_pt_...
LANGSMITH_TRACING: true
LANGSMITH_PROJECT: pc-hardware-agent

✅ LangSmith connection successful!
   Found 1 dataset(s)


In [19]:

from langsmith import Client
from langsmith.evaluation import evaluate

ls_client = Client(api_url="https://eu.api.smith.langchain.com")

# --- EVALUATION DATASET ---
# These are representative queries across all categories with expected behaviours
# We test: tool usage, output quality, scope handling, configurator triggers

dataset_name = "pc-hardware-agent-eval-v1"

# Check if dataset already exists
existing = [d for d in ls_client.list_datasets() if d.name == dataset_name]
if existing:
    dataset = existing[0]
    print(f"Using existing dataset: {dataset.name} ({dataset.id})")
else:
    dataset = ls_client.create_dataset(
        dataset_name=dataset_name,
        description="Evaluation dataset for PC Hardware Agent — Ironhack 2026"
    )

    # Create examples with inputs and expected reference outputs
    examples = [
        # --- CORE HARDWARE (should use tool, return timestamps) ---
        {
            "input": "What are the best PC cases for airflow under $150?",
            "expected": {
                "should_use_tool": True,
                "should_have_timestamps": True,
                "should_have_specs": True,
                "category": "case",
                "should_refuse": False,
            }
        },
        {
            "input": "How does the RX 7900 GRE compare to the RTX 4070 Super?",
            "expected": {
                "should_use_tool": True,
                "should_have_timestamps": True,
                "should_have_specs": True,
                "category": "gpu_comparison",
                "should_refuse": False,
            }
        },
        {
            "input": "What CPU should I get for 1440p gaming? Budget around €300",
            "expected": {
                "should_use_tool": True,
                "should_have_timestamps": True,
                "should_have_specs": True,
                "category": "cpu",
                "should_refuse": False,
            }
        },
        {
            "input": "How many watts PSU do I need for an RTX 4090 build?",
            "expected": {
                "should_use_tool": True,
                "should_have_timestamps": True,
                "category": "psu",
                "should_refuse": False,
            }
        },
        {
            "input": "What's a good B650 motherboard with WiFi?",
            "expected": {
                "should_use_tool": True,
                "should_have_timestamps": True,
                "should_have_specs": True,
                "category": "motherboard",
                "should_refuse": False,
            }
        },
        {
            "input": "DDR4 or DDR5 for gaming — is DDR5 worth it?",
            "expected": {
                "should_use_tool": True,
                "category": "ram",
                "should_refuse": False,
            }
        },
        {
            "input": "Do I need a Gen 5 NVMe SSD or is Gen 4 enough for gaming?",
            "expected": {
                "should_use_tool": True,
                "category": "storage",
                "should_refuse": False,
            }
        },

        # --- SFF / BUILD SPECIFIC ---
        {
            "input": "I want to build a small form factor gaming PC with an RTX 4080. What case and what thermal challenges?",
            "expected": {
                "should_use_tool": True,
                "should_have_timestamps": True,
                "category": "sff_build",
                "should_refuse": False,
            }
        },

        # --- APPLE / MACBOOK (must NOT refuse) ---
        {
            "input": "What's the best MacBook I can get for around $1000?",
            "expected": {
                "should_use_tool": True,
                "category": "macbook",
                "should_refuse": False,
            }
        },
        {
            "input": "Is the Mac Mini M4 good for music production with Logic Pro?",
            "expected": {
                "should_use_tool": True,
                "category": "macbook",
                "should_refuse": False,
            }
        },
        {
            "input": "My MacBook won't turn on. I think the battery died. Can you help?",
            "expected": {
                "should_use_tool": True,
                "category": "repair",
                "should_refuse": False,
            }
        },

        # --- LAPTOPS / PRE-BUILT ---
        {
            "input": "I want a gaming laptop for Fortnite and Minecraft, budget €800",
            "expected": {
                "should_use_tool": True,
                "category": "laptop",
                "should_refuse": False,
            }
        },
        {
            "input": "What pre-built gaming desktops can I get for around €700?",
            "expected": {
                "should_use_tool": True,
                "category": "prebuilt",
                "should_refuse": False,
            }
        },

        # --- EMULATION ---
        {
            "input": "I want a portable emulation machine for retro games. What should I get?",
            "expected": {
                "should_use_tool": True,
                "category": "emulation",
                "should_refuse": False,
                "should_have_legal_notice": True,
            }
        },

        # --- GAME/SOFTWARE PERFORMANCE ---
        {
            "input": "What PC specs do I need to run Hogwarts Legacy at max settings?",
            "expected": {
                "should_use_tool": True,
                "category": "game_performance",
                "should_refuse": False,
            }
        },
        {
            "input": "I use DaVinci Resolve and After Effects daily. What hardware should I prioritise?",
            "expected": {
                "should_use_tool": True,
                "category": "professional",
                "should_refuse": False,
            }
        },

        # --- BUILD GUIDE ---
        {
            "input": "I've never built a PC before. Can you walk me through the basic steps?",
            "expected": {
                "should_use_tool": True,
                "category": "build_guide",
                "should_refuse": False,
            }
        },
        {
            "input": "How do I install a new GPU in my existing PC?",
            "expected": {
                "should_use_tool": True,
                "category": "install_guide",
                "should_refuse": False,
            }
        },

        # --- CONFIGURATOR TRIGGERS ---
        {
            "input": "I want to build a gaming PC for €1500 at 1440p. Where do I start?",
            "expected": {
                "should_use_tool": True,
                "category": "configurator_trigger",
                "should_refuse": False,
                "should_mention_configurator": True,
            }
        },
        {
            "input": "Tell me about the RTX 4070 Super",
            "expected": {
                "should_use_tool": True,
                "should_have_specs": True,
                "category": "component_info",
                "should_refuse": False,
            }
        },

        # --- EDGE CASES ---
        {
            "input": "I want 4K gaming for €300. What can I get?",
            "expected": {
                "should_use_tool": True,
                "category": "impossible_budget",
                "should_refuse": False,
                "should_be_honest_about_limitations": True,
            }
        },
        {
            "input": "My computer is slow. What should I do?",
            "expected": {
                "should_use_tool": True,
                "category": "vague_troubleshooting",
                "should_refuse": False,
            }
        },
        {
            "input": "Should I install Windows 11 or Linux on my new gaming PC?",
            "expected": {
                "should_use_tool": True,
                "category": "os",
                "should_refuse": False,
            }
        },

        # --- OUT OF SCOPE (should refuse gracefully) ---
        {
            "input": "What is the best restaurant in Barcelona?",
            "expected": {
                "should_use_tool": False,
                "category": "out_of_scope",
                "should_refuse": True,
            }
        },
        {
            "input": "Can you write me a Python script to scrape Amazon?",
            "expected": {
                "should_use_tool": False,
                "category": "out_of_scope",
                "should_refuse": True,
            }
        },
    ]

    for ex in examples:
        ls_client.create_example(
            inputs={"input": ex["input"]},
            outputs=ex["expected"],
            dataset_id=dataset.id,
        )

    print(f"✅ Created dataset '{dataset_name}' with {len(examples)} examples")

print(f"\nDataset: {dataset_name}")
print(f"View at: https://smith.langchain.com/datasets")


Using existing dataset: pc-hardware-agent-eval-v1 (51e10d92-75c0-47c6-bd77-85d627dad0b3)

Dataset: pc-hardware-agent-eval-v1
View at: https://smith.langchain.com/datasets


## **LangSmith** - Evaluators

In [20]:

# --- Define the target function (wraps our agent) ---
def agent_target(inputs: dict) -> dict:
    memory.clear()
    if "input" not in inputs or not inputs["input"]:
        return {"output": "ERROR: No input provided"}
    try:
        result = executor.invoke({"input": inputs["input"]})
        return {"output": result["output"]}
    except Exception as e:
        return {"output": f"ERROR: {str(e)}"}


# --- Custom Evaluators ---
def has_youtube_timestamps(run, example) -> dict:
    output = run.outputs.get("output", "") if run.outputs else ""
    if not output or len(output) < 10:
        return {"key": "has_timestamps", "score": 0}

    has_urls = "youtube.com/watch?v=" in output and "VIDEO_ID" not in output
    return {"key": "has_timestamps", "score": 1 if has_urls else 0}


def has_quick_specs(run, example) -> dict:
    """Check if the response contains Quick Specs formatting."""
    output = run.outputs.get("output", "")
    has_specs = "Quick Specs" in output or "• " in output
    return {"key": "has_specs", "score": 1 if has_specs else 0}


def appropriate_scope(run, example) -> dict:
    """Check if the agent correctly handled scope (refused or engaged)."""
    output = run.outputs.get("output", "")
    expected = example.outputs

    should_refuse = expected.get("should_refuse", False)
    refused = "outside my area" in output.lower() or "i focus on pc hardware" in output.lower()

    if should_refuse and refused:
        return {"key": "scope_handling", "score": 1}
    elif not should_refuse and not refused:
        return {"key": "scope_handling", "score": 1}
    else:
        return {"key": "scope_handling", "score": 0}


def has_emulation_notice(run, example) -> dict:
    """Check if emulation queries include the legal disclaimer."""
    output = run.outputs.get("output", "")
    expected = example.outputs

    if expected.get("should_have_legal_notice"):
        has_notice = "legally own" in output.lower() or "rights" in output.lower()
        return {"key": "emulation_notice", "score": 1 if has_notice else 0}
    return {"key": "emulation_notice", "score": 1}  # N/A = pass


def mentions_configurator(run, example) -> dict:
    """Check if configurator-triggering queries mention the configurator."""
    output = run.outputs.get("output", "")
    expected = example.outputs

    if expected.get("should_mention_configurator"):
        mentioned = "configurator" in output.lower() or "configure" in output.lower() or "build together" in output.lower()
        return {"key": "configurator_mention", "score": 1 if mentioned else 0}
    return {"key": "configurator_mention", "score": 1}  # N/A = pass


def response_not_empty(run, example) -> dict:
    """Basic check that the agent produced a non-trivial response."""
    output = run.outputs.get("output", "")
    is_good = len(output) > 50 and "ERROR" not in output
    return {"key": "non_empty_response", "score": 1 if is_good else 0}


# --- LLM-as-Judge Evaluator ---
from langchain_openai import ChatOpenAI as JudgeLLM

judge_llm = JudgeLLM(model="gpt-4o-mini", temperature=0)

def llm_judge_quality(run, example) -> dict:
    """Use GPT-4o-mini as a judge to rate overall response quality."""
    output = run.outputs.get("output", "")
    user_query = example.inputs.get("input", "")

    if len(output) < 20:
        return {"key": "quality_score", "score": 0}

    judge_prompt = f"""Rate the quality of this PC hardware advisor response on a scale of 1-5.

User Question: {user_query}

Advisor Response: {output[:2000]}

Rate on these criteria:
1. Relevance — Does it address the user's question?
2. Accuracy — Are the technical details plausible?
3. Helpfulness — Does it give actionable advice?
4. Professionalism — Is the tone warm and professional?
5. Citations — Does it include YouTube video links when making recommendations?

Respond with ONLY a single number from 1 to 5."""

    try:
        response = judge_llm.invoke(judge_prompt)
        score_text = response.content.strip()
        score = int(score_text[0]) if score_text[0].isdigit() else 3
        return {"key": "quality_score", "score": score / 5}  # Normalize to 0-1
    except:
        return {"key": "quality_score", "score": 0.5}


print("✅ Evaluators defined. Ready to run evaluation.")



✅ Evaluators defined. Ready to run evaluation.


### **LangSmith** - Evaluation Experiment

In [21]:
print("🔬 Running evaluation experiment...")
print("   This will invoke the agent 25 times — expect ~3-5 minutes.\n")

experiment_results = evaluate(
    agent_target,
    data=dataset_name,
    evaluators=[
        has_youtube_timestamps,
        has_quick_specs,
        appropriate_scope,
        has_emulation_notice,
        mentions_configurator,
        response_not_empty,
        llm_judge_quality,
    ],
    experiment_prefix="pc-agent-v1",
    max_concurrency=2,
    client=ls_client,  # ← Forces EU endpoint
)

print("\n✅ Evaluation complete!")
print(f"View results at: https://smith.langchain.com")
print(f"Navigate to: Datasets → {dataset_name} → Experiments")

🔬 Running evaluation experiment...
   This will invoke the agent 25 times — expect ~3-5 minutes.

View the evaluation results for experiment: 'pc-agent-v1-806fcba6' at:
https://eu.smith.langchain.com/o/b1cfc463-4b69-4a7c-9787-6161557dba6e/datasets/51e10d92-75c0-47c6-bd77-85d627dad0b3/compare?selectedSessions=fb5fdd5d-2cf4-42da-9657-e54c36137d5e




0it [00:00, ?it/s]



> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...
I'm unable to assist with that. For any programming or coding-related inquiries, I recommend checking out official documentation or programming forums for guidance.

> Finished chain.


> Entering new AgentExecutor chain...
I'm here to help with PC hardware and computing questions. If you have any inquiries about building a PC, components, or software, feel free to ask!

> Finished chain.


> Entering new AgentExecutor chain...
To address a slow computer, there are several steps you can take to diagnose and potentially improve its performance. Here are some common solutions:

1. **Check for Malware**: Run a full system scan using a reputable antivirus program to ensure that malware isn't affecting performance.

2. **Manage Startup Programs**: Disable unnecessary programs that start up with your computer. You can do this through the Task Manager (Windows) or System Preferences (Mac).

3. **Free Up Disk Space

### **LangSmith** - Display Results Summary

In [22]:
print("📊 EVALUATION RESULTS SUMMARY")
print("=" * 60)
print(f"\nExperiment: {experiment_results.experiment_name}")
print(f"Dataset: {dataset_name}")
print(f"\nView full results: https://smith.langchain.com")
print("\nKey metrics to check in the LangSmith UI:")
print("  • has_timestamps — Are YouTube links present?")
print("  • has_specs — Are Quick Specs formatted correctly?")
print("  • scope_handling — Does it refuse/engage appropriately?")
print("  • emulation_notice — Legal disclaimer on emulation queries?")
print("  • configurator_mention — Configurator offered when relevant?")
print("  • non_empty_response — Did every query get a real response?")
print("  • quality_score — LLM judge overall quality (0-1)")
print("\n💡 Tips:")
print("  • Click any failing example to see the full trace")
print("  • Compare experiments after prompt changes")
print("  • Add failing traces to your dataset for regression testing")

📊 EVALUATION RESULTS SUMMARY

Experiment: pc-agent-v1-806fcba6
Dataset: pc-hardware-agent-eval-v1

View full results: https://smith.langchain.com

Key metrics to check in the LangSmith UI:
  • has_timestamps — Are YouTube links present?
  • has_specs — Are Quick Specs formatted correctly?
  • scope_handling — Does it refuse/engage appropriately?
  • emulation_notice — Legal disclaimer on emulation queries?
  • configurator_mention — Configurator offered when relevant?
  • non_empty_response — Did every query get a real response?
  • quality_score — LLM judge overall quality (0-1)

💡 Tips:
  • Click any failing example to see the full trace
  • Compare experiments after prompt changes
  • Add failing traces to your dataset for regression testing


### **Test** Images

In [24]:
# === IMAGE TEST ===
import serpapi, os
key = os.environ.get("SERPAPI_KEY")
print(f"API key present: {bool(key)}")
print(f"API key prefix: {key[:8]}..." if key else "NO KEY")

try:
    client = serpapi.Client(api_key=key)
    r = client.search(engine="google_shopping", q="AMD Ryzen 5 7600", gl="de", num=3)
    shopping = r.get("shopping_results", [])
    print(f"Shopping results: {len(shopping)}")
    if shopping:
        print(f"First thumbnail: {shopping[0].get('thumbnail', 'NONE')}")
    else:
        print(f"Keys returned: {list(r.keys())}")
except Exception as e:
    print(f"ERROR: {type(e).__name__}: {e}")

API key present: True
API key prefix: 8e29a982...
ERROR: HTTPError: 429 Client Error: Too Many Requests for url: https://serpapi.com/search?engine=google_shopping&q=AMD+Ryzen+5+7600&gl=de&num=3&api_key=8e29a982e641817da8e654d220d10fc38b6d1f8bedfa8d42fd460cc910fb4314


## **BackEnd** - FastAPI

> Note: below is testing from google colab.



#####   **Colab** (FastAPI backend)  ▶  **ngrok** (tunnel url)  ▶  **Loveable**   (frontend)  

In [32]:


!pip install fastapi uvicorn pyngrok nest_asyncio --quiet

import threading
import uvicorn
import nest_asyncio
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional, Dict, List
import uuid
import re



import time
time.sleep(1)

nest_asyncio.apply()

# Clear memory from test cells for clean API sessions
memory.clear()
print("🧹 Conversation memory cleared for fresh API start.")

# --- Create the API ---
api = FastAPI(title="PC Hardware Agent API", version="2.1.0")

api.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# --- Session store ---
api_sessions: Dict[str, Dict] = {}

def get_or_create_session(session_id: Optional[str] = None) -> str:
    sid = session_id or str(uuid.uuid4())[:8]
    if sid not in api_sessions:
        api_sessions[sid] = {
            "configurator": {
                "active": False,
                "use_case": None,
                "budget": None,
                "form_factor": None,
                "components": {
                    "case": None, "cpu": None, "gpu": None,
                    "motherboard": None, "ram": None, "storage": None,
                    "psu": None, "cooling": None, "monitor": None, "os": None,
                },
                "constraints": [],
                "notes": [],
            },
            "pending_notifications": [],
        }
    return sid


# --- Models (ONE of each, no duplicates) ---

class ChatRequest(BaseModel):
    message: str
    session_id: Optional[str] = None

class ChatResponse(BaseModel):
    response: str
    session_id: str
    configurator_active: bool = False
    configurator_state: Optional[Dict] = None
    mentioned_products: List[str] = []
    product_images: Dict[str, str] = {}
    search_sources: List[str] = []
    display_mode: str = "selection"
    next_steps: List[Dict] = []
    builds: List[Dict] = []

class ConfigAction(BaseModel):
    session_id: str
    action: str
    slot: Optional[str] = None
    value: Optional[str] = None
    constraints: Optional[List[str]] = None
    use_case: Optional[str] = None
    budget: Optional[int] = None
    form_factor: Optional[str] = None

class DetailRequest(BaseModel):
    product_name: str
    session_id: str


# --- Extraction & Parsing Functions ---

def extract_mentioned_products(text: str) -> list:
    """Extract product names from hidden tag or bold-text fallback."""
    tag_match = re.search(r'<!--\s*PRODUCTS:\s*(.+?)\s*-->', text)
    if tag_match:
        return [p.strip() for p in tag_match.group(1).split('|') if p.strip()]

    bold_matches = re.findall(r'\*\*([^*]{3,80})\*\*', text)
    PRODUCT_KEYWORDS = [
        'RTX', 'GTX', 'RX ', 'Arc ', 'Radeon', 'GeForce',
        'Ryzen', 'Core i', 'i3-', 'i5-', 'i7-', 'i9-', 'Threadripper',
        '4000D', '5000D', 'H5', 'H7', 'Meshify', 'O11', 'NR200',
        'Torrent', 'Pop', 'Meshlicious', 'Dan',
        'DDR4', 'DDR5', 'Vengeance', 'Trident', 'Fury', 'Ripjaws',
        'SN770', 'SN850', 'T700', 'T500', '990 Pro', '980 Pro', '870 EVO',
        'Noctua', 'NH-', 'Dark Rock', 'Hyper 212', 'Kraken', 'Thermalright',
        'RM750', 'RM850', 'RM1000', 'SuperNOVA', 'Focus',
        'B650', 'B550', 'X670', 'X570', 'Z790', 'Z690', 'B760',
        'MacBook', 'ThinkPad', 'XPS', 'Razer Blade', 'ROG', 'Legion',
        'Zephyrus', 'Mac Mini', 'Mac Studio', 'iMac',
        'Envy', 'Spectre', 'Inspiron', 'Latitude', 'ZenBook', 'Swift',
        'Nitro', 'Predator', 'Surface',
        'Odyssey', 'UltraGear', 'Alienware',
        'G Pro', 'DeathAdder', 'Viper', 'K70', 'Huntsman', 'Wooting',
        'Crucial P3', 'Samsung 970', 'Samsung 990', 'WD Blue', 'WD Black',
        'EVGA', 'Seasonic', 'Corsair RM', 'be quiet',
    ]
    EXCLUDED = [
        'Quick Specs', 'At a Glance', 'Suited For', 'Build Difficulty',
        'Ready to Use', 'Upgrade Potential', 'Install', 'Repair',
        'Requirements', 'Current Selections', 'Side-by-Side',
        'PC Configurator', 'What would you like', 'Your core build',
        'Performance Overview', 'Gaming Performance', 'Summary',
        'Video Insights', 'Specifications', 'Frame Rates',
        'From online sources',
    ]
    products = []
    seen = set()
    for match in bold_matches:
        m = match.strip()
        if any(e.lower() in m.lower() for e in EXCLUDED):
            continue
        if any(k.lower() in m.lower() for k in PRODUCT_KEYWORDS):
            if m.lower() not in seen:
                seen.add(m.lower())
                products.append(m)
    return products


def extract_display_mode(text: str) -> str:
    """Extract display mode from hidden tag. Defaults to 'selection'."""
    match = re.search(r'<!--\s*DISPLAY:\s*(\w+)\s*-->', text)
    if match:
        mode = match.group(1).lower()
        if mode in ("selection", "research"):
            return mode
    # Fallback: if 2+ products mentioned, it's selection; otherwise research
    products = extract_mentioned_products(text)
    return "selection" if len(products) >= 2 else "research"


def parse_configurator_selections(text: str) -> dict:
    """Parse hidden component selection and config tags."""
    updates = {"components": {}, "config": {}}

    VALID_SLOTS = {"case", "cpu", "gpu", "motherboard", "ram",
                   "storage", "psu", "cooling", "monitor", "os"}

    for slot, value in re.findall(r'<!--\s*SELECTED:\s*(\w+)=(.+?)\s*-->', text):
        slot = slot.lower().strip()
        if slot in VALID_SLOTS and value.strip():
            updates["components"][slot] = value.strip()

    for key, value in re.findall(r'<!--\s*CONFIG:\s*(\w+)=(.+?)\s*-->', text):
        key = key.lower().strip()
        value = value.strip()
        if key == "budget":
            try:
                updates["config"][key] = int(re.sub(r'[^\d]', '', value))
            except ValueError:
                pass
        elif key in ("use_case", "form_factor"):
            updates["config"][key] = value

    return updates

def parse_next_steps(text: str) -> list:
    """Parse next component suggestions from bot response."""
    steps = []
    for match in re.findall(r'---NEXT:\s*(.+?)\s*---\s*(.+?)(?=\n|---)', text):
        slot = match[0].strip()
        reason = match[1].strip().rstrip('.')
        steps.append({"slot": slot.lower(), "reason": reason})
    return steps

######### SerpAPI Now Expired this is the fall back! ############# VVV

def _get_component_image(slot: str, product_name: str) -> str:
    """Return a product image using known reliable URLs by brand/slot."""
    name_lower = product_name.lower()

    # Known product images (Wikimedia Commons — public, no auth, CORS-friendly)
    BRAND_IMAGES = {
        # GPUs
        "rtx": "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a4/NVIDIA_logo.svg/200px-NVIDIA_logo.svg.png",
        "gtx": "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a4/NVIDIA_logo.svg/200px-NVIDIA_logo.svg.png",
        "geforce": "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a4/NVIDIA_logo.svg/200px-NVIDIA_logo.svg.png",
        "radeon": "https://upload.wikimedia.org/wikipedia/commons/thumb/7/7c/AMD_Logo.svg/200px-AMD_Logo.svg.png",
        "rx ": "https://upload.wikimedia.org/wikipedia/commons/thumb/7/7c/AMD_Logo.svg/200px-AMD_Logo.svg.png",
        "arc ": "https://upload.wikimedia.org/wikipedia/commons/thumb/6/6a/Intel_logo_%282020%2C_dark_blue%29.svg/200px-Intel_logo_%282020%2C_dark_blue%29.svg.png",
        # CPUs
        "ryzen": "https://upload.wikimedia.org/wikipedia/commons/thumb/7/7c/AMD_Logo.svg/200px-AMD_Logo.svg.png",
        "threadripper": "https://upload.wikimedia.org/wikipedia/commons/thumb/7/7c/AMD_Logo.svg/200px-AMD_Logo.svg.png",
        "core i": "https://upload.wikimedia.org/wikipedia/commons/thumb/6/6a/Intel_logo_%282020%2C_dark_blue%29.svg/200px-Intel_logo_%282020%2C_dark_blue%29.svg.png",
        "i5-": "https://upload.wikimedia.org/wikipedia/commons/thumb/6/6a/Intel_logo_%282020%2C_dark_blue%29.svg/200px-Intel_logo_%282020%2C_dark_blue%29.svg.png",
        "i7-": "https://upload.wikimedia.org/wikipedia/commons/thumb/6/6a/Intel_logo_%282020%2C_dark_blue%29.svg/200px-Intel_logo_%282020%2C_dark_blue%29.svg.png",
        "i9-": "https://upload.wikimedia.org/wikipedia/commons/thumb/6/6a/Intel_logo_%282020%2C_dark_blue%29.svg/200px-Intel_logo_%282020%2C_dark_blue%29.svg.png",
        # Motherboards
        "asus": "https://upload.wikimedia.org/wikipedia/commons/thumb/2/2e/ASUS_Logo.svg/200px-ASUS_Logo.svg.png",
        "msi": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/13/MSI_Logo.svg/200px-MSI_Logo.svg.png",
        "gigabyte": "https://upload.wikimedia.org/wikipedia/commons/thumb/2/21/Gigabyte_Technology_logo_20080107.svg/200px-Gigabyte_Technology_logo_20080107.svg.png",
        "asrock": "https://upload.wikimedia.org/wikipedia/commons/thumb/5/5a/ASRock_logo.svg/200px-ASRock_logo.svg.png",
        # Cases
        "corsair": "https://upload.wikimedia.org/wikipedia/commons/thumb/5/55/Corsair_logo.svg/200px-Corsair_logo.svg.png",
        "fractal": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/42/Fractal_Design_logo.svg/200px-Fractal_Design_logo.svg.png",
        "nzxt": "https://upload.wikimedia.org/wikipedia/commons/thumb/0/00/NZXT_logo.svg/200px-NZXT_logo.svg.png",
        "lian li": "https://upload.wikimedia.org/wikipedia/commons/thumb/5/55/Corsair_logo.svg/200px-Corsair_logo.svg.png",
        "phanteks": "https://upload.wikimedia.org/wikipedia/commons/thumb/5/55/Corsair_logo.svg/200px-Corsair_logo.svg.png",
        "cooler master": "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Cooler_Master_Logo.svg/200px-Cooler_Master_Logo.svg.png",
        # RAM
        "kingston": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4a/Kingston_Technology_logo.svg/200px-Kingston_Technology_logo.svg.png",
        "g.skill": "https://upload.wikimedia.org/wikipedia/commons/thumb/e/e1/G.Skill_logo.svg/200px-G.Skill_logo.svg.png",
        "crucial": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/d3/Micron_Technology_logo.svg/200px-Micron_Technology_logo.svg.png",
        "vengeance": "https://upload.wikimedia.org/wikipedia/commons/thumb/5/55/Corsair_logo.svg/200px-Corsair_logo.svg.png",
        "trident": "https://upload.wikimedia.org/wikipedia/commons/thumb/e/e1/G.Skill_logo.svg/200px-G.Skill_logo.svg.png",
        "ripjaws": "https://upload.wikimedia.org/wikipedia/commons/thumb/e/e1/G.Skill_logo.svg/200px-G.Skill_logo.svg.png",
        # Storage
        "samsung": "https://upload.wikimedia.org/wikipedia/commons/thumb/2/24/Samsung_Logo.svg/200px-Samsung_Logo.svg.png",
        "western digital": "https://upload.wikimedia.org/wikipedia/commons/thumb/3/37/Western_Digital_logo.svg/200px-Western_Digital_logo.svg.png",
        "wd ": "https://upload.wikimedia.org/wikipedia/commons/thumb/3/37/Western_Digital_logo.svg/200px-Western_Digital_logo.svg.png",
        "seagate": "https://upload.wikimedia.org/wikipedia/commons/thumb/a/ab/Seagate_logo.svg/200px-Seagate_logo.svg.png",
        # PSU
        "seasonic": "https://upload.wikimedia.org/wikipedia/commons/thumb/5/55/Corsair_logo.svg/200px-Corsair_logo.svg.png",
        "evga": "https://upload.wikimedia.org/wikipedia/commons/thumb/f/f6/EVGA_logo.svg/200px-EVGA_logo.svg.png",
        "be quiet": "https://upload.wikimedia.org/wikipedia/commons/thumb/5/55/Corsair_logo.svg/200px-Corsair_logo.svg.png",
        # Cooling
        "noctua": "https://upload.wikimedia.org/wikipedia/commons/thumb/5/55/Corsair_logo.svg/200px-Corsair_logo.svg.png",
        # Apple
        "macbook": "https://upload.wikimedia.org/wikipedia/commons/thumb/f/fa/Apple_logo_black.svg/200px-Apple_logo_black.svg.png",
        "mac mini": "https://upload.wikimedia.org/wikipedia/commons/thumb/f/fa/Apple_logo_black.svg/200px-Apple_logo_black.svg.png",
        "imac": "https://upload.wikimedia.org/wikipedia/commons/thumb/f/fa/Apple_logo_black.svg/200px-Apple_logo_black.svg.png",
        # OS
        "windows": "https://upload.wikimedia.org/wikipedia/commons/thumb/8/87/Windows_logo_-_2021.svg/200px-Windows_logo_-_2021.svg.png",
        "linux": "https://upload.wikimedia.org/wikipedia/commons/thumb/3/35/Tux.svg/200px-Tux.svg.png",
        "ubuntu": "https://upload.wikimedia.org/wikipedia/commons/thumb/a/ab/Logo-ubuntu_cof-orange-hex.svg/200px-Logo-ubuntu_cof-orange-hex.svg.png",
    }

    # Slot-based fallbacks (if no brand matches)
    SLOT_FALLBACKS = {
        "cpu": "https://upload.wikimedia.org/wikipedia/commons/thumb/e/ea/Cpu-pins.jpg/200px-Cpu-pins.jpg",
        "gpu": "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a4/NVIDIA_logo.svg/200px-NVIDIA_logo.svg.png",
        "motherboard": "https://upload.wikimedia.org/wikipedia/commons/thumb/5/5a/ASRock_logo.svg/200px-ASRock_logo.svg.png",
        "ram": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/d3/Micron_Technology_logo.svg/200px-Micron_Technology_logo.svg.png",
        "storage": "https://upload.wikimedia.org/wikipedia/commons/thumb/2/24/Samsung_Logo.svg/200px-Samsung_Logo.svg.png",
        "psu": "https://upload.wikimedia.org/wikipedia/commons/thumb/5/55/Corsair_logo.svg/200px-Corsair_logo.svg.png",
        "case": "https://upload.wikimedia.org/wikipedia/commons/thumb/0/00/NZXT_logo.svg/200px-NZXT_logo.svg.png",
        "cooling": "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Cooler_Master_Logo.svg/200px-Cooler_Master_Logo.svg.png",
        "monitor": "https://upload.wikimedia.org/wikipedia/commons/thumb/2/24/Samsung_Logo.svg/200px-Samsung_Logo.svg.png",
        "os": "https://upload.wikimedia.org/wikipedia/commons/thumb/8/87/Windows_logo_-_2021.svg/200px-Windows_logo_-_2021.svg.png",
    }

    # Match by brand keywords
    for keyword, url in BRAND_IMAGES.items():
        if keyword in name_lower:
            print(f"  ✅ IMG: {slot} → {keyword} brand match")
            return url

    # Fallback by slot type
    if slot in SLOT_FALLBACKS:
        print(f"  ✅ IMG: {slot} → slot fallback")
        return SLOT_FALLBACKS[slot]


    # Ultimate fallback — generic image by slot type
    generic = SLOT_FALLBACKS.get(slot, "https://upload.wikimedia.org/wikipedia/commons/thumb/e/ea/Cpu-pins.jpg/200px-Cpu-pins.jpg")
    print(f"  ⚠️ IMG: {slot} → generic fallback")
    return generic

def parse_build_configurations(text: str) -> list:
    """
    Parse complete build configurations from bot response.

    Fixes vs original:
    - Handles decimal prices: ~€64.95 (not just integers)
    - Handles bold markdown: **~€782.00** in Total row
    - Adds image_url per component via Google Images (SerpAPI)
    - Includes 'os' and 'monitor' slots (not just 8 core slots)
    - Falls back gracefully if SerpAPI fails
    """
    builds = []

    build_pattern = (
        r'---BUILD\s+(\d+):\s*(.+?)\s*---'
        r'(.+?)'
        r'(?=---BUILD|---END BUILDS---|---END OPTIONS---|$)'
    )
    matches = re.findall(build_pattern, text, re.DOTALL)

    valid_slots = {
        "case", "cpu", "gpu", "motherboard", "ram",
        "storage", "psu", "cooling", "os", "monitor"
    }

    for number, name, content in matches:
        build = {
            "number": int(number),
            "name": name.strip(),
            "content": content.strip(),
            "components": {},
            "total_price": None,
        }

        # ── Parse component table rows ──────────────────────────────
        # Handles:  | Case  | Zalman I3 Neo V2  | ~€64.95 |
        # Handles:  | **Total** | | **~€782.00** |
        table_rows = re.findall(
            r'\|\s*\*{0,2}([\w][\w\s]*?)\*{0,2}\s*'
            r'\|\s*(.+?)\s*'
            r'\|\s*\*{0,2}~?€?([\d]+\.?\d*)\*{0,2}\s*\|',
            content
        )

        for slot, product_raw, price in table_rows:
            slot_lower = slot.lower().strip()
            if slot_lower not in valid_slots:
                continue

            # Handle 4-column tables: "Product Name | Key Specs" → split on pipe
            parts = product_raw.split("|")
            product_clean = parts[0].strip()
            key_specs = parts[1].strip() if len(parts) > 1 else ""

            # Clean markdown bold from product name
            product_clean = re.sub(r'\*{1,2}', '', product_clean).strip()
            # Skip separator/header rows
            if not product_clean or product_clean.startswith("-") or product_clean.startswith("="):
                continue

            print(f"  🔍 Fetching image: {slot_lower} → {product_clean}")
            image_url = _get_component_image(slot_lower, product_clean)

            build["components"][slot_lower] = {
                "product": product_clean,
                "price": float(price),
                "image_url": image_url,
                "key_specs": key_specs,
            }

        # ── Parse total price ───────────────────────────────────────
        # Handles: | **Total** | | **~€782.00** |
        total_match = re.search(
            r'\*{0,2}Total\*{0,2}.+?\*{0,2}~?€?([\d]+\.?\d*)\*{0,2}',
            content, re.IGNORECASE
        )
        if total_match:
            build["total_price"] = float(total_match.group(1))

        builds.append(build)
        print(f"  ✅ Build {number} parsed: {len(build['components'])} components, total €{build['total_price']}")

    return builds

def fallback_detect_selections(response: str) -> dict:
    """Fallback: detect selections from conversational confirmation patterns."""
    components = {}
    lower = response.lower()

    # --- Pattern 1: "✅ [Product Name] added to your build" ---
    added_match = re.search(
        r'✅\s*\*{0,2}(.+?)\*{0,2}\s+added to your build',
        response, re.IGNORECASE
    )
    if added_match:
        product_name = added_match.group(1).strip()
        slot = _detect_slot_from_product(product_name)
        if slot:
            components[slot] = product_name
            return components

    # --- Pattern 2: "Great choice! The [Product] is..." or "Going with [Product]" ---
    choice_patterns = [
        r'(?:great|excellent|good|nice|solid)\s+choice[!.]?\s*(?:the\s+)?\*{0,2}(.+?)\*{0,2}\s+(?:is|will|offers)',
        r'(?:going with|you\'ve chosen|i\'ll add|let\'s add)\s+(?:the\s+)?\*{0,2}(.+?)\*{0,2}',
        r'✅\s*\*{0,2}(.+?)\*{0,2}\s*(?:added|selected|chosen)',
    ]
    for pattern in choice_patterns:
        match = re.search(pattern, response, re.IGNORECASE)
        if match:
            product_name = match.group(1).strip()
            # Clean up trailing punctuation
            product_name = re.sub(r'[.!,]+$', '', product_name).strip()
            slot = _detect_slot_from_product(product_name)
            if slot:
                components[slot] = product_name
                return components

    # --- Pattern 3: Bold product name in a confirmation context ---
    confirmation_patterns = [
        "added to your build", "added to your", "great choice",
        "excellent choice", "good choice", "i'll add", "let's add",
        "going with", "you've chosen", "✅",
    ]
    if not any(p in lower for p in confirmation_patterns):
        return components

    bold_matches = re.findall(r'\*\*([^*]{3,80})\*\*', response)
    bracket_matches = re.findall(r'\[([^\]]{3,80})\]', response)
    all_names = bold_matches + bracket_matches

    EXCLUDED = [
        "what's next", "quick specs", "at a glance", "performance",
        "suited for", "build difficulty", "ready to use", "compare",
        "next:", "skip", "current selections",
    ]

    for name in all_names:
        if any(e in name.lower() for e in EXCLUDED):
            continue
        slot = _detect_slot_from_product(name)
        if slot:
            components[slot] = name.strip()
            return components

    return components


def _detect_slot_from_product(product_name: str) -> str:
    """Detect which configurator slot a product belongs to."""
    name_lower = product_name.lower()

    # Order matters — more specific matches first
    SLOT_RULES = [
        ("gpu", ["rtx ", "gtx ", "rx ", "radeon", "geforce", "arc a", "arc b"]),
        ("cpu", ["ryzen", "core i", "i3-", "i5-", "i7-", "i9-", "threadripper",
                 "5600g", "5700g", "5800x", "5900x", "7800x", "9800x", "9950x",
                 "14600k", "13600k", "12700k", "14900k"]),
        ("motherboard", ["motherboard", "b550", "b650", "b760", "x670", "x570",
                         "z790", "z690", "a620", "b450"]),
        ("ram", ["ddr4", "ddr5", "vengeance", "trident", "ripjaws", "fury",
                 "16gb", "32gb", "64gb"]),
        ("storage", ["ssd", "nvme", "sn770", "sn850", "990 pro", "980 pro",
                     "870 evo", "crucial p", "wd blue", "wd black", "t700",
                     "500gb", "1tb", "2tb"]),
        ("psu", ["psu", "power supply", "rm750", "rm850", "rm1000",
                 "supernova", "focus gx", "550w", "650w", "750w", "850w", "1000w"]),
        ("cooling", ["noctua", "nh-", "dark rock", "hyper 212", "kraken",
                     "thermalright", "arctic", "cooler", "aio", "240mm", "280mm", "360mm"]),
        ("case", ["case", "4000d", "5000d", "meshify", "o11", "nr200",
                  "torrent", "pop", "meshlicious", "dan a4", "h5", "h7",
                  "fractal", "lian li", "phanteks", "corsair 4", "corsair 5",
                  "nzxt", "cooler master", "antec", "in win"]),
        ("monitor", ["monitor", "odyssey", "ultragear", "27g", "32g"]),
        ("os", ["windows 11", "windows 10", "linux", "ubuntu"]),
    ]

    for slot, keywords in SLOT_RULES:
        if any(kw in name_lower for kw in keywords):
            return slot

    return ""


def clean_response_for_frontend(text: str) -> str:
    text = re.sub(r'\s*<!--\s*PRODUCTS:.*?-->\s*', '', text)
    text = re.sub(r'\s*<!--\s*SELECTED:.*?-->\s*', '', text)
    text = re.sub(r'\s*<!--\s*CONFIG:.*?-->\s*', '', text)
    text = re.sub(r'\s*<!--\s*DISPLAY:.*?-->\s*', '', text)
    text = re.sub(r'---NEXT:.*?---.*', '', text)
    text = re.sub(r'---END NEXT---', '', text)
    return text.strip()


# --- Configurator auto-activation ---

CONFIGURATOR_TRIGGERS = [
    # Exact phrases from bot responses
    "configurator activated",
    "let's start configuring",
    "start configuring your",
    "begin building your",
    "configure a pc build",
    "let's configure",
    "begin configuring",
    "configuring your custom pc",
    "custom pc build",
    "let's get started on your pc build",
    "tailor the configuration",
    "let's configure a system together",
    # Casual user triggers
    "build a pc",
    "build a computer",
    "build pc",
    "pc build",
    "configure a system",
    "let's build",
    "start a build",
    "lets make a computer",
    "make a computer",
    "make a pc",
]
def check_configurator_activation(text: str) -> bool:
    lower = text.lower()
    return any(t in lower for t in CONFIGURATOR_TRIGGERS)


# --- Endpoints ---

@api.get("/health")
async def health():
    return {
        "status": "healthy",
        "chroma_count": col.count(),
        "sessions": len(api_sessions),
    }


@api.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest):
    """Main chat endpoint."""
    sid = get_or_create_session(req.session_id)
    session_config = api_sessions[sid]["configurator"]

    # Sync session state to global configurator
    for key in ["active", "use_case", "budget", "form_factor",
                 "components", "constraints", "notes"]:
        pc_configurator[key] = session_config[key]
    rebuild_agent_with_state()

    # Inject pending sidebar notifications
    pending = api_sessions[sid].get("pending_notifications", [])
    user_message = req.message
    if pending:
        context = " | ".join(pending)
        user_message = f"[SIDEBAR UPDATE: {context}] {req.message}"
        api_sessions[sid]["pending_notifications"] = []

    # Run the agent
    try:
        result = executor.invoke({"input": user_message})
        raw_response = result["output"]
    except Exception as e:
        # Retry once on failure
        try:
            memory.clear()
            rebuild_agent_with_state()
            result = executor.invoke({"input": req.message})
            raw_response = result["output"]
        except Exception as e2:
            raw_response = (
                "I encountered an issue processing your request. "
                f"Please try again. ({str(e2)[:100]})"
            )

    # --- Parse everything from the raw response ---

    # 1. Products (for "Show me more" buttons)
    # 1. Products (for "Show me more" buttons) + fetch images
    products = extract_mentioned_products(raw_response)

    # 6. Clean response for frontend
    response_text = clean_response_for_frontend(raw_response)

    # Parse next component suggestions
    next_steps = parse_next_steps(raw_response)

    builds = parse_build_configurations(raw_response)

    # Fetch images for mentioned products (non-build queries only)
    product_images = {}
    if not builds:
        for product in products[:3]:
            img_url = _get_component_image("gpu", product)
            if img_url:
                product_images[product] = img_url

    # 2. Display mode (selection vs research)
    display_mode = extract_display_mode(raw_response)

    # 3. Search sources (for loading indicator)
    sources = ["corpus"]
    if any(ind in raw_response for ind in [
        "From online sources", "PRICING FOUND", "WEB RESULTS",
        "from online", "currently available", "current price"
    ]):
        sources.append("web")

    # 4. Auto-detect configurator activation
    if not session_config["active"]:
        if check_configurator_activation(raw_response):
            session_config["active"] = True
            print(f"🔧 Auto-activated configurator for session {sid}")

    # 5. Parse component selections (tag-based + fallback)
    if session_config["active"]:
        selections = parse_configurator_selections(raw_response)

        # Fallback if no tags found
        if not selections["components"]:
            selections["components"] = fallback_detect_selections(raw_response)

        # Apply component selections
        for slot, value in selections["components"].items():
            old = session_config["components"].get(slot)
            session_config["components"][slot] = value
            print(f"  {'🔄' if old else '✅'} {slot.upper()}: {value}")

        # Apply config updates
        for key, value in selections["config"].items():
            session_config[key] = value
            print(f"  📋 {key}: {value}")


    return ChatResponse(
        response=response_text,
        session_id=sid,
        configurator_active=session_config["active"],
        configurator_state=session_config,
        mentioned_products=products,
        product_images=product_images,
        search_sources=sources,
        display_mode=display_mode,
        next_steps=next_steps,
        builds=builds,
    )


@api.post("/configurator")
async def configurator_endpoint(req: ConfigAction):
    """Manage the PC configurator."""
    sid = req.session_id
    if sid not in api_sessions:
        raise HTTPException(status_code=404, detail="Session not found")

    config = api_sessions[sid]["configurator"]

    if req.action == "activate":
        config["active"] = True
        config["components"] = {k: None for k in config["components"]}
        config["constraints"] = []
        config["notes"] = []
        config["use_case"] = req.use_case or None
        config["budget"] = req.budget or None
        config["form_factor"] = req.form_factor or None

    elif req.action == "add" and req.slot:
        if req.slot in config["components"]:
            old_value = config["components"][req.slot]
            config["components"][req.slot] = req.value
            if req.constraints:
                config["constraints"].extend(req.constraints)
            msg = f"User changed {req.slot.upper()} from {old_value} to {req.value}" if old_value else f"User selected {req.value} for {req.slot.upper()}"
            api_sessions[sid]["pending_notifications"].append(msg)

    elif req.action == "remove" and req.slot:
        if req.slot in config["components"]:
            old_value = config["components"][req.slot]
            config["components"][req.slot] = None
            if old_value:
                api_sessions[sid]["pending_notifications"].append(
                    f"User removed {req.slot.upper()} ({old_value}) from their build"
                )

    elif req.action == "reset":
        config.update({
            "active": False, "use_case": None, "budget": None,
            "form_factor": None, "constraints": [], "notes": [],
        })
        config["components"] = {k: None for k in config["components"]}
        api_sessions[sid]["pending_notifications"] = []

    elif req.action == "get":
        pass
    else:
        raise HTTPException(status_code=400, detail=f"Unknown action: {req.action}")

    selected = sum(1 for v in config["components"].values() if v)
    return {
        "success": True,
        "session_id": sid,
        "state": config,
        "summary": f"{selected}/10 components selected",
    }


@api.get("/configurator/{session_id}")
async def get_configurator_state(session_id: str):
    if session_id not in api_sessions:
        raise HTTPException(status_code=404, detail="Session not found")
    config = api_sessions[session_id]["configurator"]
    selected = sum(1 for v in config["components"].values() if v)
    return {
        "success": True,
        "session_id": session_id,
        "state": config,
        "summary": f"{selected}/10 components selected",
    }


@api.post("/detail")
async def product_detail(req: DetailRequest):
    """Get detailed product information with timestamps."""
    sid = get_or_create_session(req.session_id)
    session_config = api_sessions[sid]["configurator"]

    for key in ["active", "use_case", "budget", "form_factor",
                 "components", "constraints", "notes"]:
        pc_configurator[key] = session_config[key]
    rebuild_agent_with_state()

    detail_query = (
        f"Tell me everything about the {req.product_name}. "
        f"Full specs breakdown, what reviewers said (pros and cons), "
        f"who it's best for, competitors, and best YouTube timestamps."
    )

    try:
        result = executor.invoke({"input": detail_query})
        raw = result["output"]
        response_text = clean_response_for_frontend(raw)
    except Exception as e:
        response_text = f"Could not fetch details. ({str(e)[:100]})"

    return {
        "product_name": req.product_name,
        "detail": response_text,
        "session_id": sid,
    }


# --- Start server + ngrok ---
from pyngrok import ngrok

ngrok_token = userdata.get("NGROK_AUTH_TOKEN")
ngrok.set_auth_token(ngrok_token)

# Kill existing tunnels (free tier limit)
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)
    print(f"  Closed old tunnel: {tunnel.public_url}")


def run_server():
    uvicorn.run(api, host="0.0.0.0", port=8000, timeout_keep_alive=900)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

import time
time.sleep(2)

public_url = ngrok.connect(8000)

print(f"\n{'='*60}")
print(f"🚀 PC Hardware Agent API v2.1 is LIVE!")
print(f"{'='*60}")
print(f"\n   Public URL:    {public_url}")
print(f"   Health check:  {public_url}/health")
print(f"   API docs:      {public_url}/docs")
print(f"   Chat endpoint: {public_url}/chat")
print(f"\n   Use this URL in your Lovable frontend!")
print(f"{'='*60}")

🧹 Conversation memory cleared for fresh API start.


  Closed old tunnel: https://annelle-existential-nickie.ngrok-free.dev


INFO:     Started server process [117843]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.



🚀 PC Hardware Agent API v2.1 is LIVE!

   Public URL:    NgrokTunnel: "https://annelle-existential-nickie.ngrok-free.dev" -> "http://localhost:8000"
   Health check:  NgrokTunnel: "https://annelle-existential-nickie.ngrok-free.dev" -> "http://localhost:8000"/health
   API docs:      NgrokTunnel: "https://annelle-existential-nickie.ngrok-free.dev" -> "http://localhost:8000"/docs
   Chat endpoint: NgrokTunnel: "https://annelle-existential-nickie.ngrok-free.dev" -> "http://localhost:8000"/chat

   Use this URL in your Lovable frontend!


In [ ]:
products = [
    ("cpu", "AMD Ryzen 5 7600"),
    ("gpu", "NVIDIA GeForce RTX 4070"),
    ("case", "Fractal Design Meshify C"),
    ("motherboard", "ASUS ROG Strix B650"),
    ("gpu", "Radeon RX 7800 XT"),
    ("ram", "Corsair Vengeance 32GB DDR5"),
    ("storage", "Samsung 990 Pro 1TB"),
    ("psu", "Corsair RM750"),
    ("os", "Windows 11 Home"),
]
for slot, name in products:
    url = _get_component_image(slot, name)
    print(f"  {name} → {'✅ has URL' if url else '❌ NONE'}\n")

  ✅ IMG: cpu → ryzen brand match
  AMD Ryzen 5 7600 → ✅ has URL

  ✅ IMG: gpu → rtx brand match
  NVIDIA GeForce RTX 4070 → ✅ has URL

  ✅ IMG: case → fractal brand match
  Fractal Design Meshify C → ✅ has URL

  ✅ IMG: motherboard → asus brand match
  ASUS ROG Strix B650 → ✅ has URL

  ✅ IMG: gpu → radeon brand match
  Radeon RX 7800 XT → ✅ has URL

  ✅ IMG: ram → corsair brand match
  Corsair Vengeance 32GB DDR5 → ✅ has URL

  ✅ IMG: storage → samsung brand match
  Samsung 990 Pro 1TB → ✅ has URL

  ✅ IMG: psu → corsair brand match
  Corsair RM750 → ✅ has URL

  ✅ IMG: os → windows brand match
  Windows 11 Home → ✅ has URL



### Test Back End build theory

In [ ]:

import re, json

# ---- Step 1: Reset everything ----
reset_configurator()
memory.clear()
rebuild_agent_with_state()
print("✅ Reset complete\n")


# ---- Step 2: Simulate the Build For Me conversation ----
# This replicates a user who answered all 6 questions
# and now expects 2-3 complete builds with component tables.

SIMULATED_ANSWERS = """
I want to build a PC. Let's go with "Build it for me".

Here are my answers:
- Budget: €800
- Use case: Gaming and light streaming
- Games: Fortnite, GTA V, maybe some AAA titles
- Noise level: Quiet is nice but not essential
- Size preference: Standard mid-tower is fine
- Future upgrades: Yes, I want easy upgrades
- Anything else: No preference on brands, no RGB needed

Please now generate 2-3 complete build configurations with full component tables and prices.
"""

print("=" * 60)
print("SIMULATED USER INPUT:")
print(SIMULATED_ANSWERS)
print("=" * 60)

result = executor.invoke({"input": SIMULATED_ANSWERS})
raw_response = result["output"]

print("\n📄 RAW BOT RESPONSE:")
print(raw_response)
print("\n" + "=" * 60)


# ---- Step 3: Check for ---BUILD--- markers ----
print("\n🔍 CHECKING FOR ---BUILD--- MARKERS:")
build_markers = re.findall(r'---BUILD\s+\d+:.+?---', raw_response)
if build_markers:
    print(f"✅ Found {len(build_markers)} build marker(s):")
    for m in build_markers:
        print(f"   {m}")
else:
    print("❌ NO ---BUILD--- markers found in response!")
    print("   → Bot is not following the format. System prompt issue.")


# ---- Step 4: Check for component tables ----
print("\n🔍 CHECKING FOR COMPONENT TABLES (pipe | syntax):")
table_rows = re.findall(r'\|\s*\w[\w\s]*\s*\|.+\|', raw_response)
if table_rows:
    print(f"✅ Found {len(table_rows)} table row(s)")
    for row in table_rows[:5]:
        print(f"   {row.strip()}")
else:
    print("❌ NO table rows found!")
    print("   → Bot skipped the component table. System prompt enforcement needed.")


# ---- Step 5: Run parse_build_configurations() on the response ----
print("\n🔍 RUNNING parse_build_configurations():")
try:
    builds = parse_build_configurations(raw_response)
    if builds:
        print(f"✅ Parsed {len(builds)} build(s):")
        for b in builds:
            print(f"\n   BUILD {b['number']}: {b['name']}")
            print(f"   Total: €{b['total_price']}")
            print(f"   Components parsed: {list(b['components'].keys())}")
            if b['components']:
                for slot, info in b['components'].items():
                    print(f"     {slot}: {info['product']} (~€{info['price']})")
    else:
        print("❌ parse_build_configurations() returned empty list!")
        print("   → Regex pattern didn't match. Check marker format in raw response above.")
except NameError:
    print("❌ parse_build_configurations() not found!")
    print("   → Make sure the backend cell has been run first.")


# ---- Step 6: Summary ----
print("\n" + "=" * 60)
print("DIAGNOSIS SUMMARY")
print("=" * 60)
has_markers = bool(build_markers)
has_tables  = bool(table_rows)
has_parsed  = bool(builds) if 'builds' in dir() else False

print(f"  ---BUILD--- markers present : {'✅' if has_markers else '❌'}")
print(f"  Component tables present    : {'✅' if has_tables  else '❌'}")
print(f"  parse_build_configs works   : {'✅' if has_parsed  else '❌'}")

if not has_markers:
    print("\n⚠️  ROOT CAUSE: Bot not outputting ---BUILD--- markers.")
    print("   FIX: Strengthen system prompt enforcement (see below).")
elif not has_tables:
    print("\n⚠️  ROOT CAUSE: Bot has markers but skipped component table.")
    print("   FIX: Add 'TABLE IS MANDATORY' more prominently in prompt.")
elif not has_parsed:
    print("\n⚠️  ROOT CAUSE: Markers present but regex not matching.")
    print("   FIX: Print raw marker section and check whitespace/newlines.")
else:
    print("\n✅ Backend is working correctly.")
    print("   If frontend isn't showing tables → frontend rendering bug.")


In [ ]:
# TEST CELL — Run this to verify images work
print("Testing image fetch...")
test_url = _get_component_image("cpu", "AMD Ryzen 5 7600")
print(f"\nResult: {test_url[:100] if test_url else 'EMPTY'}")

test_url2 = _get_component_image("case", "Fractal Design Meshify C")
print(f"Result: {test_url2[:100] if test_url2 else 'EMPTY'}")